# 🏗️ IaC Benchmark Dataset Builder
> Builds a curated Terraform dataset for LLM IaC-generation benchmarks  
> modelled after **IaC-Eval (NeurIPS 2024)**, **DPIaC-Eval**, and **Multi-IaC-Eval (arXiv 2509.05303)**.

## Pipeline overview
```
1. Clone / fetch repositories
2. Ethical licence filter
3. Collect .tf files → raw dataset
4. Size filter (LOC / token budget)
5. Syntax validation  (TFLint + HCL2 parse)
6. Deployability check (terraform init + terraform plan)
7. AWS targeting filter
8. Aggregate & export
```

### Citation
> *"We use TFLint and Checkov to ensure that all source IaC templates used as the basis of IaC mutation meet standards of IaC best practices and security."*  
> — Davidson et al., Multi-IaC-Eval (2025), §3.5


In [1]:
# ── 1. Install dependencies ────────────────────────────────────────────────────
# Run once; comment out after first execution.
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs], check=True)

pip('requests', 'python-hcl2', 'tiktoken', 'pandas', 'tqdm',
    'PyYAML', 'gitpython', 'boto3', 'tabulate', 'rich', 'matplotlib')

print('✅ Python dependencies installed.')

# ── External tools (install once on system) ────────────────────────────────────
# terraform : https://developer.hashicorp.com/terraform/downloads
# tflint    : https://github.com/terraform-linters/tflint
# checkov   : pip install checkov
# All three must be on $PATH for cells 5 and 6.
for tool in ['terraform', 'tflint', 'checkov']:
    rc = subprocess.run(['which', tool], capture_output=True).returncode
    status = '✅' if rc == 0 else '⚠️  NOT FOUND – install before running cells 5/6'
    print(f'{tool:12s} {status}')


✅ Python dependencies installed.
terraform    ✅
tflint       ✅
checkov      ✅


In [1]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
import os, pathlib

%env AWS_PROFILE=localstack
# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_DIR   = pathlib.Path('./iac_benchmark')          # root workspace
REPOS_DIR  = BASE_DIR / 'repos'                       # git clones land here
DATASET_DIR = BASE_DIR / 'dataset'                    # output CSVs
TMP_DIR    = BASE_DIR / 'tmp_plan'                    # isolated terraform plan dirs
for d in [REPOS_DIR, DATASET_DIR, TMP_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── GitHub API token (optional – raises rate limit from 60 to 5000 req/hr) ───
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '')     # set in env or paste here
GH_HEADERS   = {'Authorization': f'token {GITHUB_TOKEN}'} if GITHUB_TOKEN else {}

# ── AWS credentials (required for cell 6 – terraform plan) ───────────────────
# Best practice: export AWS_PROFILE=my-sandbox before launching Jupyter.
# A read-only IAM user with no deployment rights is sufficient for `terraform plan`.
AWS_REGION = os.environ.get('AWS_DEFAULT_REGION', 'ap-southeast-2')  # Sydney

# ── Size-filter thresholds (cell 4) ──────────────────────────────────────────
MIN_LOC    = 5        # discard near-empty stubs
MAX_LOC    = 600      # upper bound used by DPIaC-Eval (Level 5 ≈ 200 LoC, we add buffer)
MAX_TOKENS = 8_000    # ~8 k-token context budget leaves room for prompt + response

# ── Token encoder (for LLM token counting) ───────────────────────────────────
import tiktoken
TOKENIZER = tiktoken.get_encoding('cl100k_base')   # GPT-4 / Claude compatible

print('Configuration loaded.')
print(f'  BASE_DIR  : {BASE_DIR.resolve()}')
print(f'  AWS_REGION: {AWS_REGION}')
print(f'  LOC range : [{MIN_LOC}, {MAX_LOC}]  |  max tokens: {MAX_TOKENS}')


env: AWS_PROFILE=localstack
Configuration loaded.
  BASE_DIR  : /Users/iksena/Documents/research/data_analysis/iac_benchmark
  AWS_REGION: ap-southeast-2
  LOC range : [5, 600]  |  max tokens: 8000


In [2]:
# ── 3. Repository Registry ────────────────────────────────────────────────────
# Each entry carries:
#   owner/repo, clone_url, subdirectory to scan ('' = whole repo),
#   primary cloud target, and the licence string reported on GitHub.
#
# Licence eligibility follows Multi-IaC-Eval & IaC-Eval practice:
# only OSI-approved permissive licences that allow redistribution
# in a research dataset are accepted (cell 3b performs the API check).

REPO_REGISTRY = [
    dict(
        slug='hashicorp/terraform-guides',
        subdir='',
        cloud='multi',
        category='official',
        description='HashiCorp official usage patterns across providers'
    ),
    dict(
        slug='GoogleCloudPlatform/terraform-google-examples',
        subdir='',
        cloud='gcp',
        category='official',
        description='GCP examples: GKE, SQL, Vault'
    ),
    dict(
        slug='aws-samples/aws-terraform-best-practices',
        subdir='',
        cloud='aws',
        category='official',
        description='AWS architecture best practices – modularised .tf'
    ),
    dict(
        slug='terraform-aws-modules/terraform-aws-vpc',
        subdir='examples',
        cloud='aws',
        category='official',
        description='Complete, deployable VPC configurations'
    ),
    dict(
        slug='futurice/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Copy-paste recipes for AWS and Azure'
    ),
    dict(
        slug='brokedba/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='OCI, AWS, Azure, GCP, Alicloud – broad multi-cloud'
    ),
    dict(
        slug='ContainerSolutions/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Simple, idiomatic – single-file per example (ideal Pass@1)'
    ),
    dict(
        slug='ned1313/terraform-tuesdays',
        subdir='',
        cloud='multi',
        category='educational',
        description='Weekly demos for AWS and Azure'
    ),
    dict(
        slug='alfonsof/terraform-aws-examples',
        subdir='',
        cloud='aws',
        category='specialised',
        description='Step-by-step from single server to full cluster + LB + ASG'
    ),
    dict(
        slug='alfonsof/terraform-azure-examples',
        subdir='',
        cloud='azure',
        category='specialised',
        description='Azure VMs, NSGs, resource management'
    ),
    dict(
        slug='databricks/terraform-databricks-examples',
        subdir='',
        cloud='multi',
        category='specialised',
        description='Databricks data-platform across AWS, Azure, GCP'
    ),
    dict(
        slug='aws-cloudformation/iac-model-evaluation',
        subdir='terraform',
        cloud='aws',
        category='benchmark-source',
        description='High-quality IaC samples specifically curated for LLM evaluation, excluding training data'
    ),
    dict(
        slug='autoiac-project/iac-eval',
        subdir='dataset/terraform',
        cloud='aws',
        category='benchmark-source',
        description='A human-curated dataset of 458 Terraform questions ranging from simple to complex'
    ),
    dict(
        slug='gitmurali/terraform-aws-snippets',
        subdir='',
        cloud='aws',
        category='educational',
        description='Clean, single-directory snippets for VPC, RDS with Secrets Manager, and ECS Fargate'
    ),
    dict(
        slug='garutilorenzo/aws-terraform-examples',
        subdir='examples',
        cloud='aws',
        category='educational',
        description='Provisioning AWS resources using simple main.tf entrypoints for modules'
    ),
    dict(
        slug='ValeriiVasianovych/terraform-iac',
        subdir='',
        cloud='aws',
        category='educational',
        description='Learning resource with practical single-file examples for AWS infrastructure tasks'
    ),
    dict(
        slug='antonputra/tutorials',
        subdir='lessons/040',
        cloud='aws',
        category='educational',
        description='Beginner-friendly single-file examples for EC2, IAM, and Remote State configuration'
    ),
    dict(
        slug='ovotech/terraform-testing-samples',
        subdir='',
        cloud='aws',
        category='best-practices',
        description='Production-grade code samples focusing on IaC testing and verification steps'
    ),
    dict(
        slug='zealvora/terraform-beginner-to-advanced-resource',
        subdir='',
        cloud='aws',
        category='educational',
        description='Course resources with atomic .tf files for EC2, VPC, and Security Groups'
    ),
    dict(
        slug='wardviaene/terraform-course',
        subdir='',
        cloud='aws',
        category='educational',
        description='Demo-based single-file configurations for ELB, AutoScaling, and VPC'
    ),
    dict(
        slug='diodonfrost/terraform-aws-examples',
        subdir='',
        cloud='aws',
        category='educational',
        description='A collection of independent AWS manifests for fast deployment testing'
    ),
    dict(
        slug='brikis98/terraform-up-and-running-code',
        subdir='code/terraform/02-intro-to-terraform-syntax',
        cloud='aws',
        category='educational',
        description='Reference code for single-file Terraform syntax and basic AWS resources'
    ),
    dict(
        slug='aws-samples/cict-terraform-scripts',
        subdir='terraform',
        cloud='aws',
        category='best-practices',
        description='AWS-curated scripts demonstrating verified root-module patterns'
    ),
    dict(
        slug='stack72/terraform-examples',
        subdir='',
        cloud='multi',
        category='educational',
        description='Simple, single-template examples covering AWS, Azure, and core providers'
    ),
    dict(
        slug='Tianyi2/IRIS',
        subdir='',
        cloud='multi',
        category='benchmark-source',
        description='Benchmark for IaC static analysis'
    )
]

print(f'Registered {len(REPO_REGISTRY)} repositories.')


Registered 25 repositories.


In [3]:
# ── 3b. Ethical Licence Filter ────────────────────────────────────────────────
# Query the GitHub Licence API for each repository.
# Only OSI-approved permissive licences are kept in the pipeline.
# Copyleft (GPL) and proprietary licences are flagged and excluded.
#
# Results are cached to DATASET_DIR/licence_cache.csv so subsequent runs
# skip the GitHub API entirely.  Delete the file to force a fresh check.

import pandas as pd
import requests, json
from rich.console import Console
from rich.table import Table

LICENCE_CACHE = DATASET_DIR / 'licence_cache.csv'

# OSI-approved licences permissible in a research / redistribution context
ALLOWED_SPDX = {
    'MIT', 'MIT-0', 'Apache-2.0', 'BSD-2-Clause', 'BSD-3-Clause',
    'ISC', 'MPL-2.0', 'CC0-1.0', 'Unlicense', 'WTFPL'
}
# Copyleft / viral licences – included in benchmark ONLY if licence explicitly
# permits use in academic datasets (check repo NOTICE/README manually)
COPYLEFT_SPDX = {'GPL-2.0', 'GPL-3.0', 'LGPL-2.1', 'LGPL-3.0', 'AGPL-3.0'}

def fetch_licence(slug: str) -> dict:
    url = f'https://api.github.com/repos/{slug}/license'
    r = requests.get(url, headers=GH_HEADERS, timeout=15)
    if r.status_code == 200:
        data = r.json()
        spdx = data.get('license', {}).get('spdx_id', 'NOASSERTION')
        name = data.get('license', {}).get('name', 'Unknown')
        return {'spdx': spdx, 'name': name}
    elif r.status_code == 404:
        return {'spdx': 'NONE', 'name': 'No licence file found'}
    else:
        return {'spdx': 'ERROR', 'name': f'HTTP {r.status_code}'}

# ── Load from cache if available, otherwise hit GitHub API ───────────────────
if LICENCE_CACHE.exists():
    df_licence_cache = pd.read_csv(LICENCE_CACHE, index_col='slug')
    _cache = df_licence_cache.to_dict('index')   # {slug: {spdx, name, eligible, note}}
    print(f'✅ Loaded licence cache from {LICENCE_CACHE} ({len(_cache)} entries)')
    print('   Delete the file to force a fresh GitHub API check.\n')
else:
    _cache = {}
    print('No licence cache found — querying GitHub API…\n')

console = Console()
table = Table(title='Licence Eligibility Check', show_lines=True)
table.add_column('Repository',  style='cyan')
table.add_column('SPDX',        style='yellow')
table.add_column('Eligible?',   style='bold')
table.add_column('Source')
table.add_column('Notes')

APPROVED_REPOS = []
EXCLUDED_REPOS = []
cache_rows     = []

for repo in REPO_REGISTRY:
    slug = repo['slug']

    # ── Use cache hit or fetch live ───────────────────────────────────────────
    if slug in _cache:
        cached      = _cache[slug]
        spdx        = cached['spdx']
        name        = cached['name']
        data_source = '📂 cache'
    else:
        lic         = fetch_licence(slug)
        spdx        = lic['spdx']
        name        = lic['name']
        data_source = '🌐 API'

    repo['licence_spdx'] = spdx
    repo['licence_name'] = name

    if spdx in ALLOWED_SPDX:
        eligible = '✅ YES'
        note     = 'Permissive – clear for redistribution'
        repo['licence_eligible'] = True
        APPROVED_REPOS.append(repo)
    elif spdx in COPYLEFT_SPDX:
        eligible = '⚠️  COPYLEFT'
        note     = 'Review NOTICE/README for dataset clause'
        repo['licence_eligible'] = True
        APPROVED_REPOS.append(repo)
    else:
        eligible = '❌ EXCLUDED'
        note     = f'No/unknown licence ({spdx})'
        repo['licence_eligible'] = False
        EXCLUDED_REPOS.append(repo)

    table.add_row(slug, spdx, eligible, data_source, note)

    cache_rows.append({
        'slug'     : slug,
        'spdx'     : spdx,
        'name'     : name,
        'eligible' : repo['licence_eligible'],
        'note'     : note,
    })

console.print(table)
print(f'\nApproved: {len(APPROVED_REPOS)} | Excluded: {len(EXCLUDED_REPOS)}')
print('\nManually override repo["licence_eligible"] = True for copyleft repos')
print('after confirming that the licence allows academic dataset redistribution.')

# ── Save / refresh cache CSV ──────────────────────────────────────────────────
df_new_cache = pd.DataFrame(cache_rows).set_index('slug')
df_new_cache.to_csv(LICENCE_CACHE)
print(f'\n💾 Licence cache saved → {LICENCE_CACHE}')

✅ Loaded licence cache from iac_benchmark/dataset/licence_cache.csv (25 entries)
   Delete the file to force a fresh GitHub API check.



                                             Licence Eligibility Check                                             
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Repository                          ┃ SPDX        ┃ Eligible?   ┃ Source   ┃ Notes                              ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ hashicorp/terraform-guides          │ MPL-2.0     │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ GoogleCloudPlatform/terraform-goog… │ Apache-2.0  │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ aws-samples/aws-terraform-best-pra… │ NOASSERTION │ ❌ EXCLUDED │ 📂 cache │ No/unknown licence (NOASSERTION)   │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ terraform-aws-modules/terraform-aw… │ Apache-2.0  │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ futurice/terraform-examples         │ MIT         │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ brokedba/terraform-examples         │ GPL-3.0     │ ⚠️  COPYLEFT │ 📂 cache │ Review NOTICE/README for dataset   │
│                                     │             │             │          │ clause                             │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ ContainerSolutions/terraform-examp… │ NONE        │ ❌ EXCLUDED │ 📂 cache │ No/unknown licence (NONE)          │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ ned1313/terraform-tuesdays          │ MIT         │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ alfonsof/terraform-aws-examples     │ MIT         │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ alfonsof/terraform-azure-examples   │ MIT         │ ✅ YES      │ 📂 cache │ Permissive – clear for             │
│                                     │             │             │          │ redistribution                     │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ databricks/terraform-databricks-ex… │ NOASSERTION │ ❌ EXCLUDED │ 📂 cache │ No/unknown licence (NOASSERTION)   │
├─────────────────────────────────────┼─────────────┼─────────────┼──────────┼────────────────────────────────────┤
│ aws-cloudformation/iac-model-evalu… │ MIT-0       │ ✅ YES      │ 📂 cache │


Approved: 19 | Excluded: 6

Manually override repo["licence_eligible"] = True for copyleft repos
after confirming that the licence allows academic dataset redistribution.

💾 Licence cache saved → iac_benchmark/dataset/licence_cache.csv


In [4]:
# ── 3c. Merge IaC-Eval (HuggingFace) into REPO_REGISTRY corpus ───────────────
#
# FIX (vs original): The original cell wrote each Reference output directly to
# SINGLE_ROOT_DIR and set is_merged=False/file_count=1 unconditionally, bypassing
# the single-root normalisation and size-filter steps.
#
# Correct approach:
#   1. Load & deduplicate as before.
#   2. Build IAC_EVAL_HF_RECORDS with the df_raw schema (NOT the df_roots/normalised
#      schema) so the records enter the pipeline at cell-06 (size filter) and
#      carry difficulty assigned there — NOT pre-assigned here.
#   3. Do NOT write files to SINGLE_ROOT_DIR here; that is cell-05b's job.
#   4. Do NOT pre-assign difficulty; cell-06 assigns it uniformly for all rows.
#   5. Remove the GitHub clone entry from REPO_REGISTRY as before.
#
# The IAC_EVAL_HF_RECORDS list is consumed by cell-05c which injects rows into
# df_raw AFTER cell-05b has built the normalised single-root set.

import pathlib, json, re, hashlib
import pandas as pd
import pyarrow.parquet as pq

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text))
BASE_DIR   = pathlib.Path('./iac_benchmark') 
# ── Paths ──────────────────────────────────────────────────────────────────────
IAC_EVAL_HF_DIR = BASE_DIR / 'iac_eval_hf'

_parquet_files = sorted((IAC_EVAL_HF_DIR / 'data').glob('*.parquet'))
_jsonl_file    = IAC_EVAL_HF_DIR / 'data' / 'train.jsonl'

if _parquet_files:
    df_iac_eval = pd.concat(
        [pq.read_table(str(f)).to_pandas() for f in _parquet_files],
        ignore_index=True
    )
    _fmt = f'Parquet ({len(_parquet_files)} shard(s))'
elif _jsonl_file.exists():
    df_iac_eval = pd.read_json(_jsonl_file, lines=True)
    _fmt = 'JSONL'
else:
    raise FileNotFoundError(
        f"IaC-Eval dataset not found under {IAC_EVAL_HF_DIR}/data/\n"
        "Download with:\n"
        "  from datasets import load_dataset\n"
        "  ds = load_dataset('autoiac-project/iac-eval')\n"
        "  ds['train'].to_parquet('./iac_benchmark/iac_eval_hf/data/train.parquet')\n"
        "Or export from: https://huggingface.co/datasets/autoiac-project/iac-eval/viewer/default"
    )

print(f'Loaded IaC-Eval from {_fmt}: {len(df_iac_eval):,} rows')
print(f'Columns: {list(df_iac_eval.columns)}')

# ── Column aliases ─────────────────────────────────────────────────────────────
_CODE_COL   = 'Reference output'
_PROMPT_COL = 'Prompt'
_DIFF_COL   = 'Difficulty'
_INTENT_COL = 'Intent'
_REGO_COL   = 'Rego intent'
_RES_COL    = 'Resource'

_required = {_CODE_COL, _PROMPT_COL, _DIFF_COL}
_missing  = _required - set(df_iac_eval.columns)
if _missing:
    raise ValueError(
        f"Expected columns missing from IaC-Eval dataset: {_missing}\n"
        f"Available: {list(df_iac_eval.columns)}"
    )

# ── Build records in the df_RAW schema (NOT df_roots/normalised schema) ────────
# Each Reference output is a single-file Terraform template → file_path acts as
# a synthetic relative path;  parent_dir = '' so cell-05b treats it as a
# one-file root and keeps it as-is (no merging needed).
# difficulty and dest_file are intentionally ABSENT here — they are assigned by
# cell-06 (size filter) and cell-05b (normalisation) respectively.

IAC_EVAL_HF_RECORDS = []

for idx, row in df_iac_eval.iterrows():
    content = str(row.get(_CODE_COL) or '').strip()
    if not content:
        continue

    resource_raw = str(row.get(_RES_COL, f'task_{idx}')).strip()
    safe_stem    = re.sub(r'[^\w\-]', '_', resource_raw)[:50] or f'task_{idx}'
    # Unique synthetic path that mimics a repo file_path
    synthetic_path = f'autoiac-project__iac-eval/{safe_stem}_{idx}.tf'

    loc    = content.count('\n') + 1
    tokens = count_tokens(content)

    IAC_EVAL_HF_RECORDS.append({
        # ── df_raw schema (matches cell-06 / cell-05b input) ───────────────────
        'source_slug'     : 'autoiac-project/iac-eval',
        'source_category' : 'benchmark-source',
        'primary_cloud'   : 'aws',
        'licence_spdx'    : 'MIT',
        'file_path'       : synthetic_path,
        'github_url'      : 'https://huggingface.co/datasets/autoiac-project/iac-eval/viewer/default',
        'content'         : content,
        'loc'             : loc,
        'tokens'          : tokens,
        # parent_dir = '' → cell-05b sees this as a single-file root; no merge
        'parent_dir'      : '',

        # ── IaC-Eval-native bonus columns ──────────────────────────────────────
        'iac_eval_resource' : str(row.get(_RES_COL,    '')).strip(),
        'iac_eval_prompt'   : str(row.get(_PROMPT_COL, '')).strip(),
        'iac_eval_intent'   : str(row.get(_INTENT_COL, '')).strip(),
        'iac_eval_rego'     : str(row.get(_REGO_COL,   '')).strip(),
        'iac_eval_level'    : str(row.get(_DIFF_COL,   '')).strip().lower(),
    })

df_iac_eval_raw = pd.DataFrame(IAC_EVAL_HF_RECORDS)

# ── Deduplicate on content hash ────────────────────────────────────────────────
df_iac_eval_raw['_hash'] = df_iac_eval_raw['content'].apply(
    lambda c: hashlib.md5(c.encode()).hexdigest()
)
before_dedup    = len(df_iac_eval_raw)
df_iac_eval_raw = (df_iac_eval_raw
    .drop_duplicates(subset='_hash', keep='first')
    .drop(columns='_hash')
    .reset_index(drop=True)
)
IAC_EVAL_HF_RECORDS = df_iac_eval_raw.to_dict('records')

print(f'\nNormalised {before_dedup:,} IaC-Eval rows')
print(f'Duplicates removed (identical content): {before_dedup - len(df_iac_eval_raw):,}')
print(f'Unique templates retained            : {len(df_iac_eval_raw):,}')
print(f'\nLevel distribution (original label):')
print(df_iac_eval_raw['iac_eval_level'].value_counts().to_string())
print(f'\nTop 10 resources:')
print(df_iac_eval_raw['iac_eval_resource'].value_counts().head(10).to_string())

# ── Deregister the GitHub clone of the same repo ──────────────────────────────
_DEREGISTER_SLUG = 'autoiac-project/iac-eval'
_before = len(REPO_REGISTRY)
REPO_REGISTRY = [r for r in REPO_REGISTRY if r['slug'] != _DEREGISTER_SLUG]
print(f'\nDeregistered "{_DEREGISTER_SLUG}" from REPO_REGISTRY '
      f'({_before} → {len(REPO_REGISTRY)} repos) to avoid double-counting.')


Loaded IaC-Eval from Parquet (1 shard(s)): 458 rows
Columns: ['Resource', 'Prompt', 'Rego intent', 'Difficulty', 'Reference output', 'Intent']

Normalised 458 IaC-Eval rows
Duplicates removed (identical content): 90
Unique templates retained            : 368

Level distribution (original label):
iac_eval_level
2    86
3    85
6    55
5    52
4    50
1    40

Top 10 resources:
iac_eval_resource
aws_db_instance                                                                                                                13
aws_instance, aws_lb, aws_lb_listener, aws_lb_target_group, aws_lb_target_group_attachment, aws_subnet, aws_subnet, aws_vpc    10
aws_dynamodb_table                                                                                                             10
aws_lightsail_instance                                                                                                          6
aws_lex_bot, aws_lex_intent                                                        

In [5]:
# ── 4. Clone / Update Repositories ───────────────────────────────────────────
# Uses shallow clone (depth=1) to save disk space.
# If the directory already exists, pulls the latest changes instead.

import git
from tqdm.notebook import tqdm

def clone_or_pull(slug: str, dest: pathlib.Path) -> bool:
    url = f'https://github.com/{slug}.git'
    try:
        if dest.exists():
            r = git.Repo(dest)
            r.remotes.origin.pull(depth=1)
            return True
        else:
            git.Repo.clone_from(url, dest, depth=1, single_branch=True)
            return True
    except Exception as e:
        print(f'  ⚠️  {slug}: {e}')
        return False

eligible = [r for r in REPO_REGISTRY if r.get('licence_eligible')]
print(f'Cloning {len(eligible)} approved repositories…\n')

for repo in tqdm(eligible, desc='Cloning'):
    slug   = repo['slug']
    dest   = REPOS_DIR / slug.replace('/', '__')
    repo['local_path'] = str(dest)
    ok = clone_or_pull(slug, dest)
    print(f'  {"✅" if ok else "❌"} {slug}')

print('\nClone phase complete.')


Cloning 18 approved repositories…



Cloning:   0%|          | 0/18 [00:00<?, ?it/s]

  ✅ hashicorp/terraform-guides
  ✅ GoogleCloudPlatform/terraform-google-examples
  ✅ terraform-aws-modules/terraform-aws-vpc
  ✅ futurice/terraform-examples
  ✅ brokedba/terraform-examples
  ✅ ned1313/terraform-tuesdays
  ✅ alfonsof/terraform-aws-examples
  ✅ alfonsof/terraform-azure-examples
  ✅ aws-cloudformation/iac-model-evaluation
  ✅ gitmurali/terraform-aws-snippets
  ⚠️  garutilorenzo/aws-terraform-examples: Cmd('git') failed due to: exit code(128)
  cmdline: git pull -v --depth=1 -- origin
  stderr: 'fatal: Need to specify how to reconcile divergent branches.'
  ❌ garutilorenzo/aws-terraform-examples
  ✅ ValeriiVasianovych/terraform-iac
  ✅ antonputra/tutorials
  ✅ ovotech/terraform-testing-samples
  ⚠️  diodonfrost/terraform-aws-examples: Cmd('git') failed due to: exit code(128)
  cmdline: git pull -v --depth=1 -- origin
  stderr: 'fatal: Need to specify how to reconcile divergent branches.'
  ❌ diodonfrost/terraform-aws-examples
  ✅ brikis98/terraform-up-and-running-code
  ✅ 

In [6]:
# # ── 4b. ALTERNATIVE – Fetch .tf files via GitHub API (no disk clone) ─────────
# # Useful when you only want to sample files without cloning multi-GB repos.
# # Skip this cell if you used cell 4 (clone).

# import requests, base64, time

# def api_get(url: str) -> dict | list | None:
#     r = requests.get(url, headers=GH_HEADERS, timeout=20)
#     if r.status_code == 403:                     # rate limited
#         reset = int(r.headers.get('X-RateLimit-Reset', time.time() + 60))
#         wait  = max(reset - int(time.time()), 5)
#         print(f'Rate-limited – waiting {wait}s…')
#         time.sleep(wait)
#         return api_get(url)
#     return r.json() if r.ok else None

# def list_tf_files(slug: str, path: str = '', ref: str = 'HEAD') -> list[dict]:
#     """Recursively list all .tf files in a GitHub repo subtree."""
#     url   = f'https://api.github.com/repos/{slug}/git/trees/{ref}?recursive=1'
#     tree  = api_get(url)
#     if not tree:
#         return []
#     files = [
#         item for item in tree.get('tree', [])
#         if item['type'] == 'blob'
#         and item['path'].endswith('.tf')
#         and (path == '' or item['path'].startswith(path))
#     ]
#     return files

# def fetch_file_content(slug: str, sha: str) -> str:
#     """Fetch raw file content by blob SHA."""
#     url  = f'https://api.github.com/repos/{slug}/git/blobs/{sha}'
#     blob = api_get(url)
#     if not blob:
#         return ''
#     return base64.b64decode(blob['content']).decode('utf-8', errors='replace')

# # Example: fetch files from one repo without cloning
# # slug = 'ContainerSolutions/terraform-examples'
# # files = list_tf_files(slug)
# # content = fetch_file_content(slug, files[0]['sha'])
# print('API-fetch helpers defined. Use list_tf_files() / fetch_file_content() as needed.')


In [7]:
# ── 5. Collect .tf Files → Raw Dataset ───────────────────────────────────────
# Walks approved cloned repos and harvests every .tf file.
# Records: path, source slug, cloud target, content, LOC, tokens.

import pandas as pd

RAW_RECORDS = []

for repo in eligible:
    if 'local_path' not in repo:
        continue
    root = pathlib.Path(repo['local_path'])
    subdir = repo.get('subdir', '')
    scan_root = root / subdir if subdir else root

    for tf_path in scan_root.rglob('*.tf'):
        try:
            content = tf_path.read_text(encoding='utf-8', errors='replace')
        except Exception:
            continue

        loc    = content.count('\n') + 1
        tokens = count_tokens(content)
        # Relative path within repo (for reproducibility)
        rel    = str(tf_path.relative_to(root))

        RAW_RECORDS.append({
            'source_slug'       : repo['slug'],
            'source_category'   : repo['category'],
            'primary_cloud'     : repo['cloud'],
            'licence_spdx'      : repo['licence_spdx'],
            'file_path'         : rel,
            'github_url'        : f'https://github.com/{repo["slug"]}/blob/HEAD/{rel}',
            'content'           : content,
            'loc'               : loc,
            'tokens'            : tokens,
        })

df_raw = pd.DataFrame(RAW_RECORDS)
print(f'Raw collection: {len(df_raw):,} .tf files')
print(df_raw[['source_slug','primary_cloud','loc','tokens']]
      .groupby('source_slug').agg({'loc':'mean', 'tokens':'mean', 'primary_cloud':'first'})
      .round(1).to_string())


Raw collection: 117,940 .tf files
                                                 loc  tokens primary_cloud
source_slug                                                               
GoogleCloudPlatform/terraform-google-examples  103.4   610.9           gcp
Tianyi2/IRIS                                    54.9   418.8         multi
ValeriiVasianovych/terraform-iac                28.8   201.6           aws
alfonsof/terraform-aws-examples                 19.2   125.2           aws
alfonsof/terraform-azure-examples               52.6   376.1         azure
antonputra/tutorials                            13.2    73.2           aws
aws-cloudformation/iac-model-evaluation         35.2   256.7           aws
aws-samples/cict-terraform-scripts              35.9   227.5           aws
brikis98/terraform-up-and-running-code          46.6   263.9           aws
brokedba/terraform-examples                     70.0   597.4         multi
diodonfrost/terraform-aws-examples              31.0   209.0      

In [8]:
# # ── 5b. Single-Root Template Normalisation ────────────────────────────────────
# # Groups collected .tf files by parent directory (= one Terraform root),
# # merges multi-file roots into a single self-contained string, normalises
# # all whitespace variants so duplicates that differ only in newlines/spaces
# # are detected and removed before any downstream step.

# import hashlib, re, shutil
# from pathlib import Path

# SINGLE_ROOT_DIR = BASE_DIR / 'single_root_templates'
# SINGLE_ROOT_DIR.mkdir(parents=True, exist_ok=True)

# # ── Tunables ──────────────────────────────────────────────────────────────────
# MAX_FILES_PER_ROOT = 10    # roots with more files are likely framework modules
# MERGE_MULTI_FILE   = True  # False → keep only single-file roots

# # ── Normalisation ─────────────────────────────────────────────────────────────
# # Applied to EVERY file's content BEFORE merging and BEFORE hashing.
# # Order matters: unify endings first, then expand tabs, then strip trailing
# # whitespace per line, then collapse excess blank lines, then trim file edges.

# def normalise_hcl(text: str) -> str:
#     """
#     Normalise HCL content so whitespace-only variants produce identical output.
#     Semantic content (comments, values, identifiers) is never altered.
#     """
#     # 1. Unify line endings (CRLF → LF, lone CR → LF)
#     text = text.replace('\r\n', '\n').replace('\r', '\n')
#     # 2. Expand tabs to 2 spaces (HCL/Terraform formatter convention)
#     text = text.replace('\t', '  ')
#     # 3. Strip trailing whitespace on every line
#     text = '\n'.join(line.rstrip() for line in text.split('\n'))
#     # 4. Collapse 3+ consecutive blank lines → exactly 2
#     text = re.sub(r'\n{3,}', '\n\n', text)
#     # 5. Strip leading/trailing blank lines from the whole file
#     return text.strip()

# def content_hash(text: str) -> str:
#     """SHA-256 of already-normalised content."""
#     return hashlib.sha256(text.encode('utf-8')).hexdigest()

# # ── Skip-path predicate ───────────────────────────────────────────────────────
# _SKIP_RE = re.compile(
#     r'(^|/)(\.|__pycache__|\.git|\.terraform|\.terragrunt-cache'
#     r'|tests?|fixtures?|testdata|vendor|node_modules)(/|$)',
#     re.IGNORECASE,
# )

# def _should_skip(rel_path: str) -> bool:
#     return bool(_SKIP_RE.search(rel_path))

# # ── Pre-normalise every raw record's content ──────────────────────────────────
# n_before = len(df_raw)
# print(f"▶ Before normalisation  : {n_before:,} raw .tf records")

# df_raw = df_raw[~df_raw['file_path'].apply(_should_skip)].copy()
# n_after_skip = len(df_raw)
# print(f"  Skipped (test/fixture): {n_before - n_after_skip:,}")
# print(f"  Remaining             : {n_after_skip:,}")

# # Normalise in-place — downstream merge uses 'content_norm' not 'content'
# df_raw['content_norm'] = df_raw['content'].apply(normalise_hcl)

# # ── Group by (source_slug, parent_dir) ────────────────────────────────────────
# df_raw['parent_dir'] = df_raw['file_path'].apply(lambda p: str(Path(p).parent))

# groups = df_raw.groupby(['source_slug', 'parent_dir'])
# print(f"\n▶ Unique Terraform roots: {len(groups):,}")

# normalised_records = []
# copy_stats = {'kept_single': 0, 'merged': 0, 'skipped_large': 0, 'skipped_no_multi': 0}

# # ── Canonical file-sort order within a merged root ────────────────────────────
# _PRIORITY = {
#     'main.tf': 0, 'variables.tf': 1, 'outputs.tf': 2,
#     'providers.tf': 3, 'versions.tf': 4, 'locals.tf': 5,
# }

# for (slug, parent), group in tqdm(groups, desc='Normalising roots'):

#     # Skip framework-level roots that are too large to be standalone examples
#     if len(group) > MAX_FILES_PER_ROOT:
#         copy_stats['skipped_large'] += 1
#         continue

#     if len(group) == 1 or not MERGE_MULTI_FILE:
#         if len(group) > 1 and not MERGE_MULTI_FILE:
#             copy_stats['skipped_no_multi'] += 1
#             continue
#         row          = group.iloc[0]
#         # Use already-normalised content; no further whitespace work needed
#         merged_norm  = row['content_norm']
#         is_merged    = False
#         origin_files = [row['file_path']]
#         copy_stats['kept_single'] += 1
#     else:
#         ordered = sorted(
#             group.itertuples(),
#             key=lambda r: (_PRIORITY.get(Path(r.file_path).name, 99), r.file_path),
#         )
#         sections = []
#         for r in ordered:
#             fname = Path(r.file_path).name
#             # Section separator is itself a normalised single blank line
#             header = f'# ── {fname} ──────────────────────────────────────────'
#             # r.content_norm is already normalised — no re-normalisation needed
#             sections.append(f'{header}\n{r.content_norm}')
#         # Join sections with exactly ONE blank line (2 newlines) between them
#         merged_norm  = '\n\n'.join(sections)
#         # Final pass: collapse any accidental 3+ blank lines at section seams
#         merged_norm  = re.sub(r'\n{3,}', '\n\n', merged_norm).strip()
#         is_merged    = True
#         origin_files = [r.file_path for r in ordered]
#         copy_stats['merged'] += 1

#     row0        = group.iloc[0]
#     loc         = merged_norm.count('\n') + 1
#     tokens      = count_tokens(merged_norm)
#     folder_name = (parent.replace('/', '__').replace(' ', '_') or 'ROOT')
#     dest_slug   = slug.replace('/', '__')
#     chash       = content_hash(merged_norm)

#     dest_dir  = SINGLE_ROOT_DIR / dest_slug
#     dest_dir.mkdir(parents=True, exist_ok=True)
#     dest_file = dest_dir / f'{folder_name}.tf'
#     dest_file.write_text(merged_norm, encoding='utf-8')

#     normalised_records.append({
#         'source_slug'    : slug,
#         'source_category': row0['source_category'],
#         'primary_cloud'  : row0['primary_cloud'],
#         'licence_spdx'   : row0['licence_spdx'],
#         'root_path'      : str(Path(slug) / parent) if parent else slug,
#         'origin_files'   : ' | '.join(origin_files),
#         'file_count'     : len(origin_files),
#         'is_merged'      : is_merged,
#         'dest_file'      : str(dest_file.relative_to(BASE_DIR)),
#         'github_url'     : (f'https://github.com/{slug}/tree/HEAD/{parent}'
#                             if parent else f'https://github.com/{slug}'),
#         'content'        : merged_norm,   # normalised content replaces raw
#         'content_hash'   : chash,
#         'loc'            : loc,
#         'tokens'         : tokens,
#     })

# df_roots = pd.DataFrame(normalised_records)

# # ── Exact-duplicate removal (post-merge, hash on fully normalised content) ────
# n_before_dedup = len(df_roots)
# df_roots = df_roots.drop_duplicates(subset='content_hash', keep='first').copy()
# n_after_dedup  = len(df_roots)

# # ── Summary ───────────────────────────────────────────────────────────────────
# print('=' * 60)
# print('SINGLE-ROOT NORMALISATION SUMMARY')
# print('=' * 60)
# print(f"  Input .tf records          : {n_after_skip:>6,}")
# print(f"  Unique roots found         : {len(groups):>6,}")
# print(f"  ✅  Kept (single-file)     : {copy_stats['kept_single'] - copy_stats['merged']:>6,}")
# print(f"  🔀  Merged (multi-file)    : {copy_stats['merged']:>6,}")
# print(f"  ⏭️   Skipped (>{MAX_FILES_PER_ROOT} files)   : {copy_stats['skipped_large']:>6,}")
# print(f"  🗑️   Exact duplicates removed: {n_before_dedup - n_after_dedup:>6,}")
# print(f"  ▶ Output templates         : {n_after_dedup:>6,}")
# print(f"\nOutput folder : {SINGLE_ROOT_DIR.resolve()}")
# print()
# print('Top 5 sources by template count:')
# print(df_roots.groupby('source_slug')['root_path']
#       .count().sort_values(ascending=False).head(5).to_string())
# print()
# print('Merged vs single-file breakdown:')
# print(df_roots['is_merged'].value_counts()
#       .rename({True: 'merged', False: 'single-file'}).to_string())

# # ── Promote to df_raw for all downstream cells ────────────────────────────────
# df_raw = df_roots.copy()
# print('\n✅ df_raw replaced with normalised single-root dataset.')
# print('   All downstream cells will use normalised content.')

# # ── Inject IaC-Eval HF dataset rows (if collected) ────────────────────────────
# if 'IAC_EVAL_HF_RECORDS' in dir() and IAC_EVAL_HF_RECORDS:
#     df_iac_hf = (
#         pd.DataFrame(IAC_EVAL_HF_RECORDS)
#         .reindex(columns=df_raw.columns)
#     )
#     # Normalise HF content too — same pipeline, same guarantee
#     df_iac_hf['content'] = df_iac_hf['content'].apply(normalise_hcl)
#     df_iac_hf['content_hash'] = df_iac_hf['content'].apply(content_hash)
#     df_raw = pd.concat([df_raw, df_iac_hf], ignore_index=True)
#     # Remove cross-source duplicates introduced by HF injection
#     before_hf_dedup = len(df_raw)
#     df_raw = df_raw.drop_duplicates(subset='content_hash', keep='first').copy()
#     print(f'\n  + {len(df_iac_hf):,} IaC-Eval HF rows merged')
#     print(f'  - {before_hf_dedup - len(df_raw):,} cross-source duplicates removed')
#     print(f'  ▶ df_raw total: {len(df_raw):,}')

In [9]:
# ── 5b-alt. Multi-File Root Collector ────────────────────────────────────────
# ALTERNATIVE to cell-05b (Single-Root Template Normalisation).
# Comment out cell-05b and run this cell instead.
#
# Strategy: for each Terraform root directory (= one logical example), copy
# the entire folder to MULTI_ROOT_DIR and record dest_file pointing at the
# root entry-point (main.tf, or the first .tf alphabetically).  Downstream
# tools (tflint, checkov, trivy) already operate on a directory, so they will
# see all sibling files (variables.tf, outputs.tf, …) without any merging.
#
# Output schema is identical to cell-05b so ALL downstream cells are
# unaffected:
#   dest_file       → relative path of the root entry-point .tf inside BASE_DIR
#   root_path       → logical slug/parent-dir path (for provenance)
#   origin_files    → '|'-joined list of every .tf file in the root
#   file_count      → number of .tf files in this root
#   is_merged       → always False (files are NOT concatenated)
#   content         → content of dest_file only (entry-point only)
#   content_hash    → SHA-256 of normalised entry-point content
#   loc / tokens    → measured on entry-point content
#
# Input : df_raw  (from cell-06, the raw .tf collection)
# Output: df_raw  (replaced with multi-root normalised dataset)

import hashlib, re, shutil
from pathlib import Path

MULTI_ROOT_DIR = BASE_DIR / 'multi_root_templates'
MULTI_ROOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Tunables ──────────────────────────────────────────────────────────────────
MAX_FILES_PER_ROOT = 10     # roots with more .tf files are likely framework
                             # modules, not self-contained examples — skip them

# ── Reuse helpers from cell-05b (must be run first) ──────────────────────────
# normalise_hcl(), content_hash(), _should_skip() are defined in cell-05b.
# If you have fully commented out cell-05b, copy those three functions here:

# def normalise_hcl(text): ...  (copy from cell-05b if needed)
# def content_hash(text):  ...
# def _should_skip(path):  ...

# ── Preferred entry-point filenames (in priority order) ──────────────────────
_ENTRYPOINT_PRIORITY = [
    'main.tf', 'variables.tf', 'outputs.tf',
    'providers.tf', 'versions.tf', 'locals.tf',
]

def pick_entrypoint(tf_files: list[Path]) -> Path:
    """Return the most canonical root .tf file from a list."""
    names = {f.name: f for f in tf_files}
    for name in _ENTRYPOINT_PRIORITY:
        if name in names:
            return names[name]
    return sorted(tf_files)[0]   # fallback: alphabetically first

# ── Skip-path predicate (keep in sync with cell-05b) ─────────────────────────
_SKIP_RE = re.compile(
    r'(^|/)(\.|__pycache__|\.git|\.terraform|\.terragrunt-cache'
    r'|tests?|fixtures?|testdata|vendor|node_modules)(/|$)',
    re.IGNORECASE,
)
def _should_skip(rel_path: str) -> bool:
    return bool(_SKIP_RE.search(rel_path))

# ── Normalisation helpers (copy here if cell-05b is commented out) ────────────
def normalise_hcl(text: str) -> str:
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    text = text.replace('\t', '  ')
    text = '\n'.join(line.rstrip() for line in text.split('\n'))
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def content_hash(text: str) -> str:
    return hashlib.sha256(text.encode('utf-8')).hexdigest()

# ── Pre-filter: drop test/fixture paths ──────────────────────────────────────
n_before = len(df_raw)
df_raw = df_raw[~df_raw['file_path'].apply(_should_skip)].copy()
print(f'▶ Before normalisation  : {n_before:,} raw .tf records')
print(f'  Skipped (test/fixture): {n_before - len(df_raw):,}')
print(f'  Remaining             : {len(df_raw):,}')

# ── Add parent_dir column (same grouping key as cell-05b) ────────────────────
df_raw['parent_dir'] = df_raw['file_path'].apply(lambda p: str(Path(p).parent))

groups = df_raw.groupby(['source_slug', 'parent_dir'])
print(f'\n▶ Unique Terraform roots: {len(groups):,}')

# ── Main loop ─────────────────────────────────────────────────────────────────
multi_records = []
stats = {'kept': 0, 'skipped_large': 0, 'deduped': 0}
seen_hashes: set[str] = set()

for (slug, parent), group in tqdm(groups, desc='Collecting roots'):

    tf_files_in_group = list(group['file_path'])

    # Skip oversized roots (likely framework/module trees, not examples)
    if len(tf_files_in_group) > MAX_FILES_PER_ROOT:
        stats['skipped_large'] += 1
        continue

    # ── Resolve on-disk paths ─────────────────────────────────────────────────
    repo_local = Path(
        next(r for r in eligible if r['slug'] == slug)['local_path']
    )
    root_dir_abs = repo_local / parent if parent and parent != '.' else repo_local
    tf_paths_abs = [root_dir_abs / Path(fp).name for fp in tf_files_in_group]
    tf_paths_abs = [p for p in tf_paths_abs if p.exists()]

    if not tf_paths_abs:
        continue   # source files missing (clone incomplete)

    # ── Pick entry-point ──────────────────────────────────────────────────────
    entrypoint_abs = pick_entrypoint(tf_paths_abs)
    try:
        ep_content = entrypoint_abs.read_text(encoding='utf-8', errors='replace')
    except Exception:
        continue
    ep_norm  = normalise_hcl(ep_content)
    chash    = content_hash(ep_norm)

    # Deduplicate on entry-point content hash
    if chash in seen_hashes:
        stats['deduped'] += 1
        continue
    seen_hashes.add(chash)

    # ── Copy entire root directory to MULTI_ROOT_DIR ──────────────────────────
    dest_slug    = slug.replace('/', '__')
    folder_label = (parent.replace('/', '__').replace(' ', '_') or 'ROOT')
    dest_root    = MULTI_ROOT_DIR / dest_slug / folder_label
    dest_root.mkdir(parents=True, exist_ok=True)

    for tf_abs in tf_paths_abs:
        shutil.copy2(tf_abs, dest_root / tf_abs.name)

    dest_file = dest_root / entrypoint_abs.name

    row0 = group.iloc[0]
    loc    = ep_norm.count('\n') + 1
    tokens = count_tokens(ep_norm)

    multi_records.append({
        'source_slug'     : slug,
        'source_category' : row0['source_category'],
        'primary_cloud'   : row0['primary_cloud'],
        'licence_spdx'    : row0['licence_spdx'],
        'root_path'       : str(Path(slug) / parent) if parent else slug,
        'origin_files'    : ' | '.join(tf_files_in_group),
        'file_count'      : len(tf_paths_abs),
        'is_merged'       : False,                        # never concatenated
        'dest_file'       : str(dest_file.relative_to(BASE_DIR)),
        'github_url'      : (f'https://github.com/{slug}/tree/HEAD/{parent}'
                             if parent else f'https://github.com/{slug}'),
        'content'         : ep_norm,        # entry-point content only
        'content_hash'    : chash,
        'loc'             : loc,
        'tokens'          : tokens,
    })
    stats['kept'] += 1

df_roots = pd.DataFrame(multi_records)

# ── Summary ───────────────────────────────────────────────────────────────────
print('=' * 60)
print('MULTI-ROOT COLLECTION SUMMARY')
print('=' * 60)
print(f'  Input .tf records          : {len(df_raw):>6,}')
print(f'  Unique roots found         : {len(groups):>6,}')
print(f'  ✅  Roots kept             : {stats["kept"]:>6,}')
print(f'  ⏭️   Skipped (>{MAX_FILES_PER_ROOT} files)   : {stats["skipped_large"]:>6,}')
print(f'  🗑️   Exact duplicates removed: {stats["deduped"]:>6,}')
print(f'  ▶ Output templates         : {len(df_roots):>6,}')
print(f'\nOutput folder : {MULTI_ROOT_DIR.resolve()}')
print()
print('Top 5 sources by template count:')
print(df_roots.groupby('source_slug')['root_path']
      .count().sort_values(ascending=False).head(5).to_string())
print()
print('Single-file vs multi-file roots:')
print(df_roots['file_count']
      .apply(lambda n: 'single-file' if n == 1 else 'multi-file')
      .value_counts().to_string())

# ── Promote to df_raw for all downstream cells (identical to cell-05b) ────────
df_raw = df_roots.copy()
print('\n✅ df_raw replaced with multi-root collected dataset.')
print('   dest_file points to entry-point; sibling .tf files are in the same dir.')
print('   All downstream cells (5c, 6, 7, 8…) will use this dataset.')

▶ Before normalisation  : 117,940 raw .tf records
  Skipped (test/fixture): 4
  Remaining             : 117,936

▶ Unique Terraform roots: 40,469


MULTI-ROOT COLLECTION SUMMARY
  Input .tf records          : 117,936
  Unique roots found         : 40,469
  ✅  Roots kept             : 34,633
  ⏭️   Skipped (>10 files)   :    984
  🗑️   Exact duplicates removed:  4,852
  ▶ Output templates         : 34,633

Output folder : /Users/iksena/Documents/research/data_analysis/iac_benchmark/multi_root_templates

Top 5 sources by template count:
source_slug
Tianyi2/IRIS                               34441
ValeriiVasianovych/terraform-iac              64
ned1313/terraform-tuesdays                    41
brokedba/terraform-examples                   30
aws-cloudformation/iac-model-evaluation       21

Single-file vs multi-file roots:
file_count
multi-file     17601
single-file    17032

✅ df_raw replaced with multi-root collected dataset.
   dest_file points to entry-point; sibling .tf files are in the same dir.
   All downstream cells (5c, 6, 7, 8…) will use this dataset.


In [10]:
# ── 5c. Inject IaC-Eval (HuggingFace) into normalised df_raw ─────────────────
# FIX: IAC_EVAL_HF_RECORDS are now in df_raw schema (not df_roots schema), so
# they go through cell-05b normalisation. This cell now ONLY handles cross-source
# deduplication and appending — no schema coercion needed.

import hashlib

if not ('IAC_EVAL_HF_RECORDS' in dir() and IAC_EVAL_HF_RECORDS):
    print('⚠️  IAC_EVAL_HF_RECORDS not found – run cell-03c first.')
else:
    df_iac_hf_raw = pd.DataFrame(IAC_EVAL_HF_RECORDS)

    # ── Run the same single-root normalisation as cell-05b ────────────────────
    # IaC-Eval records have parent_dir='' and file_count=1, so each passes
    # through as a kept-single-file root. This ensures dest_file is set,
    # is_merged=False, and root_path is populated consistently.

    MULTI_ROOT_DIR.mkdir(parents=True, exist_ok=True)
    iac_eval_normalised = []

    for _, row in df_iac_hf_raw.iterrows():
        content   = row['content']
        loc       = content.count('\n') + 1
        tokens    = count_tokens(content)
        slug      = row['source_slug']
        dest_slug = slug.replace('/', '__')
        safe_name = pathlib.Path(row['file_path']).name  # e.g. aws_s3_bucket_0.tf

        dest_dir  = MULTI_ROOT_DIR / dest_slug
        dest_dir.mkdir(parents=True, exist_ok=True)
        dest_file = dest_dir / safe_name
        dest_file.write_text(content, encoding='utf-8')

        rec = {
            'source_slug'     : slug,
            'source_category' : row.get('source_category', 'benchmark-source'),
            'primary_cloud'   : row.get('primary_cloud', 'aws'),
            'licence_spdx'    : row.get('licence_spdx', 'MIT'),
            'root_path'       : f"{slug}/{pathlib.Path(row['file_path']).stem}",
            'origin_files'    : row['file_path'],
            'file_count'      : 1,
            'is_merged'       : False,
            'dest_file'       : str(dest_file.relative_to(BASE_DIR)),
            'github_url'      : row.get('github_url', ''),
            'content'         : content,
            'loc'             : loc,
            'tokens'          : tokens,
            # difficulty will be assigned uniformly by cell-06
            # iac_eval bonus columns preserved
            'iac_eval_resource' : row.get('iac_eval_resource', ''),
            'iac_eval_prompt'   : row.get('iac_eval_prompt', ''),
            'iac_eval_intent'   : row.get('iac_eval_intent', ''),
            'iac_eval_rego'     : row.get('iac_eval_rego', ''),
            'iac_eval_level'    : row.get('iac_eval_level', ''),
        }
        iac_eval_normalised.append(rec)

    df_iac_hf_norm = pd.DataFrame(iac_eval_normalised)

    # ── Remove any iac-eval rows already cloned from GitHub ───────────────────
    _already = df_raw['source_slug'] == 'autoiac-project/iac-eval'
    if _already.sum() > 0:
        print(f'  Removing {_already.sum():,} rows already cloned from GitHub '
              f'(autoiac-project/iac-eval) before injection…')
        df_raw = df_raw[~_already].copy()

    # ── Cross-source content dedup ─────────────────────────────────────────────
    def _md5(s): return hashlib.md5(str(s).encode()).hexdigest()
    existing_hashes       = set(df_raw['content'].apply(_md5))
    df_iac_hf_norm['_hash'] = df_iac_hf_norm['content'].apply(_md5)
    cross_dups            = df_iac_hf_norm['_hash'].isin(existing_hashes).sum()
    df_iac_hf_norm        = (df_iac_hf_norm[~df_iac_hf_norm['_hash'].isin(existing_hashes)]
                             .drop(columns='_hash')
                             .reset_index(drop=True))

    before = len(df_raw)
    df_raw = pd.concat([df_raw, df_iac_hf_norm], ignore_index=True)

    print(f'IaC-Eval HF injection complete.')
    print(f'  Cross-source duplicates removed : {cross_dups:,}')
    print(f'  Rows injected                   : {len(df_iac_hf_norm):,}')
    print(f'  df_raw before → after           : {before:,} → {len(df_raw):,}')
    print()
    print('Source breakdown:')
    print(df_raw['source_slug'].value_counts().to_string())

    # ── Re-write index.csv to include IaC-Eval rows ───────────────────────────
    # Columns to exclude from the index (content is too large; _hash already dropped)
    _EXCLUDE = ['content', 'iac_eval_rego']   # rego policies are very long

    index_path = MULTI_ROOT_DIR / 'index.csv'
    df_raw.drop(columns=[c for c in _EXCLUDE if c in df_raw.columns]) \
          .to_csv(index_path, index=False)

    print(f'\n💾 index.csv updated → {index_path}')
    print(f'   Total rows in index: {len(df_raw):,}')


IaC-Eval HF injection complete.
  Cross-source duplicates removed : 1
  Rows injected                   : 367
  df_raw before → after           : 34,633 → 35,000

Source breakdown:
source_slug
Tianyi2/IRIS                                     34441
autoiac-project/iac-eval                           367
ValeriiVasianovych/terraform-iac                    64
ned1313/terraform-tuesdays                          41
brokedba/terraform-examples                         30
aws-cloudformation/iac-model-evaluation             21
terraform-aws-modules/terraform-aws-vpc              9
hashicorp/terraform-guides                           8
GoogleCloudPlatform/terraform-google-examples        7
gitmurali/terraform-aws-snippets                     4
alfonsof/terraform-aws-examples                      3
aws-samples/cict-terraform-scripts                   2
ovotech/terraform-testing-samples                    2
garutilorenzo/aws-terraform-examples                 1

💾 index.csv updated → iac_benchmark/

In [11]:
# ── 6. Size Filter ────────────────────────────────────────────────────────────
# Mirrors the DPIaC-Eval difficulty stratification:
#   Level 1: <50 LoC / 1-2 resources
#   Level 2: 50-100 LoC
#   Level 3: 100-150 LoC
#   Level 4: 150-200 LoC
#   Level 5: >200 LoC / 12-14 resources
#
# We also enforce a MAX_TOKENS budget so templates fit within an LLM context.
# IaC-Eval (NeurIPS 2024) notes longest configs can reach 280 LoC / 24 resources.

def assign_difficulty(loc: int) -> int:
    if loc < 50:  return 1
    if loc < 100: return 2
    if loc < 150: return 3
    if loc < 200: return 4
    return 5

df_raw['difficulty'] = df_raw['loc'].apply(assign_difficulty)

mask_loc    = df_raw['loc'].between(MIN_LOC, MAX_LOC)
mask_tokens = df_raw['tokens'] <= MAX_TOKENS

df_sized = df_raw[mask_loc & mask_tokens].copy()

print(f'After size filter: {len(df_sized):,} files (from {len(df_raw):,})')
print(f'  Removed (too short <{MIN_LOC} LoC) : {(~mask_loc).sum():,}')
print(f'  Removed (too long >{MAX_LOC} LoC)  : {(df_raw["loc"] > MAX_LOC).sum():,}')
print(f'  Removed (>{MAX_TOKENS} tokens)     : {(~mask_tokens).sum():,}')
print()
print('Difficulty distribution:')
print(df_sized['difficulty'].value_counts().sort_index().to_string())


After size filter: 33,559 files (from 35,000)
  Removed (too short <5 LoC) : 1,437
  Removed (too long >600 LoC)  : 154
  Removed (>8000 tokens)     : 49

Difficulty distribution:
difficulty
1    21020
2     6827
3     2710
4     1184
5     1818


In [12]:
df_sized.head()

,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,content,content_hash,loc,tokens,iac_eval_resource,iac_eval_prompt,iac_eval_intent,iac_eval_rego,iac_eval_level,difficulty
0,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-custom-machine-types/main.tf | example...,2,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,"provider ""google"" {\n region = ""${var.region}...",3054b2f4ec1f928868415ba0084f370c6d052b4be91c0e...,111,825,NaN,NaN,NaN,NaN,NaN,3
1,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-helm/helm.tf | example-gke-k8s...,2,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,"variable ""region"" {\n default = ""us-west1""\n}...",7373b2651dcdda69ed7a753fc04cf75b6caa293a276d5a...,74,447,NaN,NaN,NaN,NaN,NaN,2
2,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-multi-region/main.tf,1,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,"variable ""region1_cluster_name"" {\n default =...",3670fe3049856bad9675562a879ba09990b673358a0398...,198,1357,NaN,NaN,NaN,NaN,NaN,4
3,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-multi-region/gke-regional/main.tf,1,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,"variable ""region"" {\n default = ""us-west1""\n}...",aa5eba09dbaa7d9c165dd2633b4022de29d151a2f2f3fc...,80,440,NaN,NaN,NaN,NaN,NaN,2
4,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-multi-region/k8s-app/main.tf,1,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,"variable ""external_ip"" {}\nvariable ""node_port...",2f77467f90d93a31f97b7249a4af5707b2ab3549c22d34...,94,387,NaN,NaN,NaN,NaN,NaN,2


In [13]:
# ── 7. Syntax Validation ──────────────────────────────────────────────────────
# Two-stage:
#   a) python-hcl2 parse  – fast in-process HCL2 check (no external tool needed)
#   b) tflint             – provider-aware lint (requires tflint on $PATH)
#
# Multi-IaC-Eval uses TFLint + Checkov; IaC-Eval uses `terraform plan`.
# We mirror the Multi-IaC-Eval approach for the static-analysis layer.
#
# Features:
#   - Cache: results are checkpointed to VALIDATION_CACHE_PATH after each file.
#            Re-running the cell resumes from where it left off.
#   - Fault-tolerance: tflint timeout / subprocess errors → scenario is SKIPPED
#                      (tflint_pass=None, tflint_error='<reason>'), not failed.

import hcl2, io, subprocess, tempfile, json, pathlib
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()

VALIDATION_CACHE_PATH = BASE_DIR / 'dataset' / 'validation_cache.csv'
TFLINT_TIMEOUT = 30   # seconds — reduced from 300 to surface hangs quickly


# ── a) python-hcl2 parse ──────────────────────────────────────────────────────
def hcl2_valid(content: str) -> tuple[bool, str]:
    try:
        hcl2.load(io.StringIO(content))
        return True, ''
    except Exception as e:
        return False, str(e)[:200]


# ── b) tflint (subprocess) ────────────────────────────────────────────────────
# Returns (pass: bool | None, error_msg: str)
#   True  → tflint found no errors
#   False → tflint found errors
#   None  → tflint could not be executed (timeout, crash, not-found) → SKIP
def tflint_check(content: str) -> tuple[bool | None, str]:
    """Write to a temp dir and run tflint --format=json."""
    with tempfile.TemporaryDirectory() as tmpd:
        tf_file = pathlib.Path(tmpd) / 'main.tf'
        tf_file.write_text(content, encoding='utf-8')
        try:
            result = subprocess.run(
                ['tflint', '--format=json', '--no-color'],
                cwd=tmpd, capture_output=True, text=True,
                timeout=TFLINT_TIMEOUT          # ← raises TimeoutExpired if exceeded
            )
        except subprocess.TimeoutExpired:
            return None, f'tflint timeout (>{TFLINT_TIMEOUT}s) — scenario skipped'
        except FileNotFoundError:
            return None, 'tflint not found on $PATH — scenario skipped'
        except Exception as e:
            return None, f'tflint subprocess error: {str(e)[:200]} — scenario skipped'

        try:
            out = json.loads(result.stdout or '{}')
            issues = out.get('issues', [])
            errors = [i for i in issues if i.get('rule', {}).get('severity') == 'error']
            if errors:
                msgs = '; '.join(e.get('message', '') for e in errors[:3])
                return False, msgs
            return True, ''
        except json.JSONDecodeError:
            # Non-JSON output (e.g. tflint init prompt, panic) → skip, don't fail
            stderr_preview = result.stderr[:200]
            return None, f'tflint non-JSON output — scenario skipped: {stderr_preview}'


TFLINT_AVAILABLE = subprocess.run(
    ['which', 'tflint'], capture_output=True
).returncode == 0


# ── Load validation cache (resume support) ───────────────────────────────────
if VALIDATION_CACHE_PATH.exists():
    cache_df = pd.read_csv(VALIDATION_CACHE_PATH, index_col='root_path')
    print(f'📂  Resuming from cache — {len(cache_df):,} rows already validated.')
else:
    cache_df = pd.DataFrame(
        columns=['root_path', 'hcl2_valid', 'hcl2_error', 'tflint_pass', 'tflint_error']
    ).set_index('root_path')
    print('🆕  No cache found — starting fresh validation.')

already_done = set(cache_df.index)
todo_mask    = ~df_sized['root_path'].isin(already_done)
df_todo      = df_sized[todo_mask].copy()
print(f'   {len(already_done):,} cached  |  {len(df_todo):,} remaining\n')


# ── Run validation row-by-row with incremental cache writes ──────────────────
new_rows = []

for _, row in tqdm(df_todo.iterrows(), total=len(df_todo), desc='Validating'):
    content   = row['content']
    root_path = row['root_path']

    # a) HCL2 parse
    hcl2_ok, hcl2_err = hcl2_valid(content)

    # b) TFLint — None means "skip this scenario", not a validation failure
    if TFLINT_AVAILABLE:
        tflint_ok, tflint_err = tflint_check(content)
    else:
        tflint_ok, tflint_err = None, 'tflint not available — skipped'

    new_rows.append({
        'root_path':    root_path,
        'hcl2_valid':   hcl2_ok,
        'hcl2_error':   hcl2_err,
        'tflint_pass':  tflint_ok,   # True | False | None
        'tflint_error': tflint_err,
    })

    # ── Incremental cache flush (every row) ──────────────────────────────────
    # Writing per-row is safe for long runs; batch every N rows if perf matters.
    new_cache_df = pd.DataFrame(new_rows).set_index('root_path')
    updated_cache = pd.concat([cache_df, new_cache_df])
    updated_cache.to_csv(VALIDATION_CACHE_PATH)


# ── Merge cache back into df_sized ───────────────────────────────────────────
final_cache = pd.read_csv(VALIDATION_CACHE_PATH, index_col='root_path')
validation_cols = ['hcl2_valid', 'hcl2_error', 'tflint_pass', 'tflint_error']
existing_validation_cols = [c for c in validation_cols if c in df_sized.columns]
if existing_validation_cols:
    df_sized = df_sized.drop(columns=existing_validation_cols)
df_sized = df_sized.merge(
    final_cache[validation_cols],
    left_on='root_path', right_index=True, how='left'
)


# ── Summary ───────────────────────────────────────────────────────────────────
n_total       = len(df_sized)
n_hcl2_pass   = df_sized['hcl2_valid'].sum()
n_tflint_pass = (df_sized['tflint_pass'] == True).sum()
n_tflint_skip = df_sized['tflint_pass'].isna().sum()
n_tflint_fail = (df_sized['tflint_pass'] == False).sum()

print(f'\nHCL2 parse   passed : {n_hcl2_pass:,} / {n_total:,}')
if TFLINT_AVAILABLE:
    print(f'TFLint       passed : {n_tflint_pass:,} / {n_total:,}')
    print(f'TFLint       failed : {n_tflint_fail:,} / {n_total:,}')
    print(f'TFLint       skipped: {n_tflint_skip:,}  (timeout / non-JSON / error)')
else:
    print('⚠️  tflint not found — tflint columns set to None (skipped)')

print(f'\n💾  Cache saved → {VALIDATION_CACHE_PATH}')

📂  Resuming from cache — 33,559 rows already validated.
   33,559 cached  |  0 remaining



Validating: 0it [00:00, ?it/s]


HCL2 parse   passed : 32,372 / 33,559
TFLint       passed : 33,558 / 33,559
TFLint       failed : 0 / 33,559
TFLint       skipped: 1  (timeout / non-JSON / error)

💾  Cache saved → iac_benchmark/dataset/validation_cache.csv


In [14]:
# ── 7b. YAML Validation for embedded YAML here-docs ──────────────────────────
# Some Terraform files embed YAML policies (e.g. aws_iam_policy inline_document,
# kubernetes_manifest). Extract and validate any <<-EOT ... EOT here-doc blocks.

# import re, yaml

# HEREDOC_RE = re.compile(
#     r'<<[-~]?(?P<tag>[A-Z_]+)\s*\n(?P<body>.*?)\n(?P=tag)',
#     re.DOTALL
# )

# def validate_embedded_yaml(content: str) -> tuple[bool, str]:
#     """Return (all_valid, first_error). Passes if no YAML here-docs present."""
#     for m in HEREDOC_RE.finditer(content):
#         body = m.group('body')
#         # Only attempt to parse if it looks like YAML (has : or - patterns)
#         if ':' not in body and not body.strip().startswith('-'):
#             continue
#         try:
#             yaml.safe_load(body)
#         except yaml.YAMLError as e:
#             return False, str(e)[:200]
#     return True, ''

# df_sized[['yaml_valid', 'yaml_error']] = (
#     df_sized['content']
#     .progress_apply(lambda c: pd.Series(validate_embedded_yaml(c)))
# )

# print(f'YAML embedded check passed: {df_sized["yaml_valid"].sum():,} / {len(df_sized):,}')

# # ── Syntax-clean filter ───────────────────────────────────────────────────────
# tflint_col   = 'tflint_pass' if TFLINT_AVAILABLE else 'hcl2_valid'
# df_syntaxok  = df_sized[
#     df_sized['hcl2_valid'] & df_sized[tflint_col].fillna(True) & df_sized['yaml_valid']
# ].copy()

df_syntaxok = df_sized.copy()

print(f'\nSyntax-clean dataset: {len(df_syntaxok):,} files')



Syntax-clean dataset: 33,559 files


In [15]:
# ── 8. AWS Targeting Filter (TFLint + HCL2 AST) ──────────────────────────────
import hcl2, io, json, hashlib, subprocess, tempfile
from pathlib import Path
from tqdm.notebook import tqdm
import concurrent.futures as _cf

tqdm.pandas()

# ── Timeout helper ────────────────────────────────────────────────────────────
def _with_timeout(fn, args=(), kwargs=None, timeout_sec=10, default=None):
    """
    Run fn(*args, **kwargs) in a daemon thread.
    Returns default if timeout_sec is exceeded.

    IMPORTANT: executor.shutdown(wait=False) is mandatory — wait=True (the
    default used by the context-manager __exit__) blocks until the thread
    finishes, which re-introduces the hang we are trying to avoid.
    """
    ex = _cf.ThreadPoolExecutor(max_workers=1)
    fut = ex.submit(fn, *args, **(kwargs or {}))
    try:
        return fut.result(timeout=timeout_sec)
    except _cf.TimeoutError:
        return default
    except Exception:
        raise
    finally:
        ex.shutdown(wait=False)   # ← THE FIX: don't block on the stuck thread

# ── Tunables ──────────────────────────────────────────────────────────────────
USE_TFLINT_AWS_PROBE  = True
HCL2_PARSE_TIMEOUT    = 10    # seconds — per .tf file hcl2.load() wall-clock limit
TFLINT_INIT_TIMEOUT   = 60    # seconds — tflint --init network call
TFLINT_PROBE_TIMEOUT  = 60    # seconds — tflint lint run per directory
DIR_DETECT_TIMEOUT    = 120   # seconds — total budget per directory (all strategies)
AWS_FILTER_CACHE_CSV  = DATASET_DIR / 'aws_filter_cache.csv'
CACHE_FLUSH_EVERY     = 41

_AWS_PROVIDER_SOURCES = {
    'hashicorp/aws',
    'aws',
    'registry.terraform.io/hashicorp/aws',
}

AWS_CACHE_COLS = [
    'hcl2_aws', 'hcl2_aws_reason',
    'tflint_aws', 'tflint_aws_reason',
    'targets_aws',
]

# ── Directory-level cache key ─────────────────────────────────────────────────
def dir_hash(scan_dir: Path) -> str:
    tf_files = sorted(scan_dir.glob('*.tf'))
    if not tf_files:
        return ''
    h = hashlib.sha256()
    for tf in tf_files:
        h.update(tf.name.encode())
        h.update(tf.read_bytes())
    return h.hexdigest()

# ── Load or create cache ───────────────────────────────────────────────────────
if AWS_FILTER_CACHE_CSV.exists():
    _aws_cache = pd.read_csv(AWS_FILTER_CACHE_CSV, index_col='dir_hash')
    for col in ['hcl2_aws', 'tflint_aws', 'targets_aws']:
        _aws_cache[col] = _aws_cache[col].astype(str).str.lower() == 'true'
    for col in ['hcl2_aws_reason', 'tflint_aws_reason']:
        _aws_cache[col] = _aws_cache[col].fillna('').astype(str)
    print(f'✅ Loaded AWS filter cache  ({len(_aws_cache):,} entries) '
          f'— cached dirs will be skipped')
else:
    _aws_cache = pd.DataFrame(columns=AWS_CACHE_COLS)
    _aws_cache.index.name = 'dir_hash'
    print('ℹ️  No existing AWS filter cache — starting fresh')

# ── Strategy A: HCL2 AST walk ─────────────────────────────────────────────────
def _hcl2_targets_aws_dir(scan_dir: Path) -> tuple[bool, str]:
    def _unwrap(val):
        if isinstance(val, list):
            return val[0] if len(val) == 1 else val
        return val

    for tf_file in sorted(scan_dir.glob('*.tf')):
        content = tf_file.read_text(encoding='utf-8', errors='replace')

        ast = _with_timeout(
            hcl2.load,
            args=(io.StringIO(content),),
            timeout_sec=HCL2_PARSE_TIMEOUT,
            default=None,
        )
        if ast is None:
            tqdm.write(f'  ⚠️  hcl2 parse timeout/error: {tf_file.name} — skipped')
            continue

        try:
            for tf_block in ast.get('terraform', []):
                for rp in tf_block.get('required_providers', []):
                    for _name, spec in rp.items():
                        spec = _unwrap(spec)
                        src  = _unwrap(spec.get('source', '')) if isinstance(spec, dict) else spec
                        if isinstance(src, str) and src.lower() in _AWS_PROVIDER_SOURCES:
                            return True, f'required_providers.source={src} [{tf_file.name}]'

            for provider_block in ast.get('provider', []):
                if 'aws' in provider_block:
                    return True, f'provider "aws" block [{tf_file.name}]'

            for block_type in ('resource', 'data'):
                for block in ast.get(block_type, []):
                    for rtype in block.keys():
                        if rtype.startswith('aws_'):
                            return True, f'{block_type}.{rtype} [{tf_file.name}]'

            for module_block in ast.get('module', []):
                for _name, cfg in module_block.items():
                    cfg = _unwrap(cfg)
                    if not isinstance(cfg, dict):
                        continue
                    src = _unwrap(cfg.get('source', ''))
                    if not isinstance(src, str):
                        try:
                            src = str(src[0]) if isinstance(src, (list, tuple)) and src else ''
                        except (TypeError, IndexError):
                            src = ''
                    if src and ('terraform-aws-modules' in src or 'aws' in src.lower()):
                        return True, f'module.source={src} [{tf_file.name}]'

        except Exception as e:
            tqdm.write(f'  ⚠️  HCL2 walk error in {tf_file.name}: {type(e).__name__}: {e}')
            continue

    return False, 'no_aws_indicators'

# ── Strategy B: TFLint --ruleset=aws probe ────────────────────────────────────
_TFLINT_HCL_AWS = """\
plugin "aws" {
  enabled = true
  version = "0.38.0"
  source  = "github.com/terraform-linters/tflint-ruleset-aws"
}
"""

_TFLINT_PLUGIN_CACHE = Path.home() / '.tflint.d' / 'plugins'

def _tflint_plugin_cached() -> bool:
    if not _TFLINT_PLUGIN_CACHE.exists():
        return False
    return any(
        p.name.startswith('tflint-ruleset-aws')
        for p in _TFLINT_PLUGIN_CACHE.rglob('tflint-ruleset-aws*')
    )

def _tflint_targets_aws_dir(scan_dir: Path) -> tuple[bool, str]:
    with tempfile.TemporaryDirectory() as tmpd:
        tmp = Path(tmpd)
        tf_files = list(scan_dir.glob('*.tf'))
        if not tf_files:
            return False, 'no_tf_files'
        for tf in tf_files:
            (tmp / tf.name).write_bytes(tf.read_bytes())
        (tmp / '.tflint.hcl').write_text(_TFLINT_HCL_AWS, encoding='utf-8')

        if not _tflint_plugin_cached():
            try:
                init = subprocess.run(
                    ['tflint', '--init', '--no-color'],
                    cwd=tmpd, capture_output=True, timeout=TFLINT_INIT_TIMEOUT,
                )
                if init.returncode != 0:
                    return False, f'tflint_init_failed_rc={init.returncode}'
            except subprocess.TimeoutExpired:
                return False, 'tflint_init_timeout'

        try:
            r = subprocess.run(
                ['tflint', '--format=json', '--no-color'],
                cwd=tmpd, capture_output=True, text=True,
                timeout=TFLINT_PROBE_TIMEOUT,
            )
        except subprocess.TimeoutExpired:
            return False, 'tflint_probe_timeout'

        if r.returncode in (0, 2):
            return True, f'tflint_aws_ruleset_rc={r.returncode}'

        stderr_lower = r.stderr.lower()
        if any(k in stderr_lower for k in ('provider not found', 'no configuration', 'unknown')):
            return False, 'tflint_no_aws_provider'

        try:
            out = json.loads(r.stdout or '{}')
            for issue in out.get('issues', []):
                if issue.get('rule', {}).get('name', '').startswith('aws_'):
                    return True, 'tflint_aws_rule_hit'
        except (json.JSONDecodeError, AttributeError):
            pass

        return False, f'tflint_rc={r.returncode}'

# ── Detect a single directory (both strategies) ───────────────────────────────
def detect_aws_dir(scan_dir: Path, primary_cloud: str) -> dict:
    if primary_cloud == 'aws':
        return dict(
            hcl2_aws=True,   hcl2_aws_reason='declared_cloud=aws',
            tflint_aws=True, tflint_aws_reason='skipped_declared_cloud',
            targets_aws=True,
        )

    hcl2_aws, hcl2_reason = _hcl2_targets_aws_dir(scan_dir)

    if USE_TFLINT_AWS_PROBE and TFLINT_AVAILABLE and not hcl2_aws:
        tflint_aws, tflint_reason = _tflint_targets_aws_dir(scan_dir)
    else:
        tflint_aws    = hcl2_aws
        tflint_reason = 'skipped_or_confirmed_by_hcl2'

    return dict(
        hcl2_aws=hcl2_aws,       hcl2_aws_reason=hcl2_reason,
        tflint_aws=tflint_aws,   tflint_aws_reason=tflint_reason,
        targets_aws=hcl2_aws or tflint_aws,
    )

# ── Build scan-dir index — hash computed ONCE per unique directory ─────────────
df_syntaxok = df_syntaxok.copy()
df_syntaxok['_scan_dir'] = df_syntaxok['dest_file'].apply(
    lambda p: str((BASE_DIR / p).parent)
)

print('🔑 Computing directory hashes (once per unique dir)…')
unique_scan_dir_paths = df_syntaxok['_scan_dir'].unique()

_scan_dir_to_hash: dict[str, str] = {}
for d in unique_scan_dir_paths:
    p = Path(d)
    _scan_dir_to_hash[d] = dir_hash(p) if p.exists() else ''

df_syntaxok['_dir_hash'] = df_syntaxok['_scan_dir'].map(_scan_dir_to_hash).fillna('')

n_missing = (df_syntaxok['_dir_hash'] == '').sum()
print(f'   {len(unique_scan_dir_paths):,} unique dirs hashed '
      f'({n_missing} missing/empty skipped)')

unique_dirs = (
    df_syntaxok[df_syntaxok['_dir_hash'] != '']
    .groupby('_dir_hash', as_index=False)
    .agg(_scan_dir=('_scan_dir', 'first'), primary_cloud=('primary_cloud', 'first'))
)

cached_mask = unique_dirs['_dir_hash'].isin(_aws_cache.index)
dirs_cached = unique_dirs[cached_mask]
dirs_new    = unique_dirs[~cached_mask].reset_index(drop=True)

print(f'\n📂 Unique scan directories  : {len(unique_dirs):,}')
print(f'   ✅ Already cached         : {len(dirs_cached):,}  (will be skipped)')
print(f'   🔍 New / changed          : {len(dirs_new):,}  (will be detected now)')
print(f'\n  Strategy A : HCL2 AST parse     (always active)')
print(f'  Strategy B : TFLint AWS ruleset  '
      f'({"active" if USE_TFLINT_AWS_PROBE and TFLINT_AVAILABLE else "SKIPPED — tflint not found or disabled"})\n')

# ── Detect new directories ─────────────────────────────────────────────────────
new_detected = 0

for loop_i, (idx, dir_row) in enumerate(tqdm(dirs_new.iterrows(), total=len(dirs_new), desc='AWS detect')):
    if loop_i == 0 or loop_i == 1:
        continue
    dhash      = dir_row['_dir_hash']
    scan_dir   = Path(dir_row['_scan_dir'])
    prim_cloud = dir_row.get('primary_cloud', '') or ''

    tqdm.write(f'  🔍 [{idx+1}/{len(dirs_new)}] {scan_dir.name}')

    try:
        result = _with_timeout(
            detect_aws_dir,
            args=(scan_dir, prim_cloud),
            timeout_sec=DIR_DETECT_TIMEOUT,
            default=None,
        )
        if result is None:
            tqdm.write(f'  ⏱️  timed out: {scan_dir.name} — marked non-AWS')
            result = dict(
                hcl2_aws=False,   hcl2_aws_reason='dir_detect_timeout',
                tflint_aws=False, tflint_aws_reason='dir_detect_timeout',
                targets_aws=False,
            )
    except Exception as e:
        tqdm.write(f'  ❌ error: {scan_dir.name}: {type(e).__name__}: {e}')
        result = dict(
            hcl2_aws=False,   hcl2_aws_reason=f'error:{type(e).__name__}',
            tflint_aws=False, tflint_aws_reason=f'error:{type(e).__name__}',
            targets_aws=False,
        )

    _aws_cache.loc[dhash] = result
    new_detected += 1

    if new_detected % CACHE_FLUSH_EVERY == 0:
        _aws_cache.to_csv(AWS_FILTER_CACHE_CSV, index=True)
        tqdm.write(f'  💾 Cache flushed ({new_detected} new entries written)')

if new_detected > 0:
    _aws_cache.to_csv(AWS_FILTER_CACHE_CSV, index=True)
    print(f'\n✅ AWS filter cache saved → {AWS_FILTER_CACHE_CSV.relative_to(BASE_DIR)}')
    print(f'   Total cache size : {len(_aws_cache):,} entries '
          f'({new_detected} new this run)')
else:
    print('\n✅ All directories were cached — nothing new to write.')

# ── Map results back to all df_syntaxok rows (via dir_hash) ───────────────────
_empty_result = dict(
    hcl2_aws=False,   hcl2_aws_reason='missing_dir',
    tflint_aws=False, tflint_aws_reason='missing_dir',
    targets_aws=False,
)
_cache_dict = _aws_cache.to_dict(orient='index')

aws_meta = (
    df_syntaxok['_dir_hash']
    .map(lambda h: _cache_dict.get(h, _empty_result))
    .apply(pd.Series)
)
aws_meta.index = df_syntaxok.index

df_syntaxok = df_syntaxok.drop(columns=['_scan_dir', '_dir_hash']).join(aws_meta)
df_aws = df_syntaxok[df_syntaxok['targets_aws']].copy()

# ── Summary ───────────────────────────────────────────────────────────────────
n_total       = len(df_syntaxok)
n_aws         = len(df_aws)
n_hcl2_hit    = df_syntaxok['hcl2_aws'].sum()
n_tflint_hit  = df_syntaxok['tflint_aws'].sum()
n_tflint_only = (~df_syntaxok['hcl2_aws'] & df_syntaxok['tflint_aws']).sum()

print('=' * 60)
print('AWS TARGETING FILTER SUMMARY')
print('=' * 60)
print(f'  Input (syntax-clean)        : {n_total:>5,}')
print(f'  ── HCL2 AST confirmed AWS   : {n_hcl2_hit:>5,}')
print(f'  ── TFLint probe confirmed   : {n_tflint_hit:>5,}')
print(f'  ── TFLint-only (AST missed) : {n_tflint_only:>5,}')
print(f'  ✅  AWS-targeting (final)   : {n_aws:>5,}')
print()
print('Reason breakdown (HCL2):')
print(df_syntaxok['hcl2_aws_reason'].value_counts().head(10).to_string())
print()
print('Cloud breakdown:')
print(df_syntaxok['primary_cloud'].value_counts().to_string())

✅ Loaded AWS filter cache  (33,198 entries) — cached dirs will be skipped
🔑 Computing directory hashes (once per unique dir)…
   33,200 unique dirs hashed (0 missing/empty skipped)

📂 Unique scan directories  : 33,200
   ✅ Already cached         : 33,198  (will be skipped)
   🔍 New / changed          : 2  (will be detected now)

  Strategy A : HCL2 AST parse     (always active)
  Strategy B : TFLint AWS ruleset  (active)



AWS detect:   0%|          | 0/2 [00:00<?, ?it/s]


✅ All directories were cached — nothing new to write.
AWS TARGETING FILTER SUMMARY
  Input (syntax-clean)        : 33,559
  ── HCL2 AST confirmed AWS   : 11,517
  ── TFLint probe confirmed   : 30,600
  ── TFLint-only (AST missed) : 19,083
  ✅  AWS-targeting (final)   : 30,600

Reason breakdown (HCL2):
hcl2_aws_reason
no_aws_indicators                                        21498
provider "aws" block [main.tf]                            1103
provider "aws" block [main_gen.tf]                         600
declared_cloud=aws                                         465
resource.aws_instance [main.tf]                            456
error:UnexpectedCharacters                                 423
required_providers.source=hashicorp/aws [main_gen.tf]      338
required_providers.source=hashicorp/aws [main.tf]          308
resource.aws_s3_bucket [main.tf]                           197
resource.aws_iam_role [main.tf]                            177

Cloud breakdown:
primary_cloud
multi    33087
aws

In [16]:
print(len(df_aws))
df_aws.head()

30600


,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,...,difficulty,hcl2_valid,hcl2_error,tflint_pass,tflint_error,hcl2_aws,hcl2_aws_reason,tflint_aws,tflint_aws_reason,targets_aws
1,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-helm/helm.tf | example-gke-k8s...,2,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,...,2,True,NaN,True,NaN,False,no_aws_indicators,True,tflint_aws_ruleset_rc=2,True
5,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-multi-region/k8s-nginx/main.tf,1,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,...,2,True,NaN,True,NaN,False,no_aws_indicators,True,tflint_aws_ruleset_rc=2,True
6,GoogleCloudPlatform/terraform-google-examples,official,gcp,Apache-2.0,GoogleCloudPlatform/terraform-google-examples/...,example-gke-k8s-service-lb/main.tf | example-g...,2,False,multi_root_templates/GoogleCloudPlatform__terr...,https://github.com/GoogleCloudPlatform/terrafo...,...,2,True,NaN,True,NaN,False,no_aws_indicators,True,tflint_aws_ruleset_rc=2,True
7,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,3,True,NaN,True,NaN,True,"provider ""aws"" block [main.tf]",True,skipped_or_confirmed_by_hcl2,True
8,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,4,True,NaN,True,NaN,True,"provider ""aws"" block [main.tf]",True,skipped_or_confirmed_by_hcl2,True


In [17]:
# ── 8b. Checkov Security Scan + Violation Annotation ─────────────────────────
# Runs Checkov against every AWS-filtered template from df_aws.
# Violation detail (top failing check IDs + resource types) is extracted from
# the same JSON response — no second subprocess call needed.
#
# Input  : df_aws  (all rows have dest_file → BASE_DIR/single_root_templates/…)
# Output : df_checkov  (df_aws + checkov_passed, checkov_failed, checkov_skipped,
#                       checkov_score, checkov_ok,
#                       checkov_top_violations, checkov_violation_resources)

import subprocess, json
from collections import Counter
from pathlib import Path

# ── Tunables ──────────────────────────────────────────────────────────────────
CHECKOV_TIMEOUT       = 300      # seconds per template
CHECKOV_MIN_PASS      = 0.50    # used for checkov_ok flag; final gate is 0.80 (cell 8e)
CHECKOV_MAX_VIOLATIONS = 20      # top-N failing check IDs to store per template
CHECKOV_CACHE_CSV     = DATASET_DIR / 'checkov_cache.csv'

# ── Result cache ──────────────────────────────────────────────────────────────
if CHECKOV_CACHE_CSV.exists():
    _ck_cache = pd.read_csv(CHECKOV_CACHE_CSV, index_col='content_hash')
    print(f'✅ Loaded Checkov cache ({len(_ck_cache):,} entries)')
else:
    _ck_cache = pd.DataFrame(
        columns=['checkov_passed', 'checkov_failed', 'checkov_skipped',
                 'checkov_score', 'checkov_ok',
                 'checkov_top_violations', 'checkov_violation_resources']
    )
    _ck_cache.index.name = 'content_hash'

# ── Per-template scanner ──────────────────────────────────────────────────────
def run_checkov(row) -> dict:
    """
    Run `checkov -d <dir> --framework terraform --output json` and return a
    flat metrics dict including violation annotations.
    --compact is intentionally omitted so results.failed_checks is present.
    """
    chash = row.get('content_hash', '')

    # ── Cache hit ─────────────────────────────────────────────────────────────
    if chash and chash in _ck_cache.index:
        cached = _ck_cache.loc[chash]
        return {
            'checkov_passed'             : int(cached['checkov_passed']),
            'checkov_failed'             : int(cached['checkov_failed']),
            'checkov_skipped'            : int(cached['checkov_skipped']),
            'checkov_score'              : float(cached['checkov_score']),
            'checkov_ok'                 : bool(cached['checkov_ok']),
            'checkov_top_violations'     : str(cached.get('checkov_top_violations', '')),
            'checkov_violation_resources': str(cached.get('checkov_violation_resources', '')),
        }

    dest_file = BASE_DIR / row['dest_file']
    scan_dir  = dest_file.parent

    # ── Guard ─────────────────────────────────────────────────────────────────
    if not dest_file.exists():
        return dict(checkov_passed=0, checkov_failed=0, checkov_skipped=0,
                    checkov_score=0.0, checkov_ok=False,
                    checkov_top_violations='', checkov_violation_resources='')

    try:
        proc = subprocess.run(
            [
                'checkov', '-d', str(scan_dir),
                '--framework', 'terraform',
                '--output', 'json',
                '--quiet',      # suppress non-JSON stderr noise
                # --compact intentionally omitted: we need failed_checks detail
            ],
            capture_output=True,
            text=True,
            timeout=CHECKOV_TIMEOUT,
        )
        # Checkov exits 1 when checks fail — expected; only exit 2+ is an error
        raw = proc.stdout.strip()
        if not raw:
            raise ValueError('empty stdout')
        data = json.loads(raw)

        # Checkov JSON can be a list (multiple runners) or a single dict
        if isinstance(data, list):
            tf_results = next(
                (d for d in data if d.get('check_type') == 'terraform'), data[0]
            ) if data else {}
        else:
            tf_results = data

        summary = tf_results.get('summary', {})
        passed  = int(summary.get('passed',  0))
        failed  = int(summary.get('failed',  0))
        skipped = int(summary.get('skipped', 0))
        total   = passed + failed
        score   = round(passed / total, 4) if total > 0 else 1.0
        ok      = score >= CHECKOV_MIN_PASS

        # ── Violation annotation (same response, no extra subprocess) ─────────
        failed_checks    = tf_results.get('results', {}).get('failed_checks', [])
        check_counter    = Counter(c['check_id'] for c in failed_checks)
        resource_counter = Counter(
            c.get('resource', '').split('.')[0] for c in failed_checks
        )
        top_violations = '|'.join(
            cid for cid, _ in check_counter.most_common(CHECKOV_MAX_VIOLATIONS)
        )
        top_resources  = '|'.join(
            rt for rt, _ in resource_counter.most_common(CHECKOV_MAX_VIOLATIONS)
        )

    except subprocess.TimeoutExpired:
        passed, failed, skipped, score, ok = 0, 0, 0, 0.0, False
        top_violations, top_resources      = '', ''
    except Exception:
        passed, failed, skipped, score, ok = 0, 0, 0, 0.0, False
        top_violations, top_resources      = '', ''

    result = dict(
        checkov_passed=passed, checkov_failed=failed, checkov_skipped=skipped,
        checkov_score=score,   checkov_ok=ok,
        checkov_top_violations=top_violations,
        checkov_violation_resources=top_resources,
    )

    if chash:
        _ck_cache.loc[chash] = result

    return result

# ── Batch run ─────────────────────────────────────────────────────────────────
checkov_rows = []
for _, row in tqdm(df_aws.iterrows(), total=len(df_aws), desc='Checkov scan'):
    checkov_rows.append(run_checkov(row))

_ck_cache.to_csv(CHECKOV_CACHE_CSV, index=True)
print(f'\n✅ Checkov cache saved → {CHECKOV_CACHE_CSV.relative_to(BASE_DIR)}')

# ── Attach columns ────────────────────────────────────────────────────────────
df_checkov_meta = pd.DataFrame(checkov_rows, index=df_aws.index)
df_checkov      = df_aws.join(df_checkov_meta)

# ── Summary ───────────────────────────────────────────────────────────────────
n_total   = len(df_checkov)
n_ok      = df_checkov['checkov_ok'].sum()
n_fail    = n_total - n_ok
avg_score = df_checkov['checkov_score'].mean()

print('=' * 60)
print('CHECKOV SECURITY SCAN SUMMARY')
print('=' * 60)
print(f'  Input (AWS-filtered)          : {n_total:>6,}')
print(f'  ✅  Pass (score ≥ {CHECKOV_MIN_PASS:.0%})       : {n_ok:>6,}')
print(f'  ❌  Fail (score < {CHECKOV_MIN_PASS:.0%})       : {n_fail:>6,}')
print(f'  📊  Average pass-rate          : {avg_score:>8.1%}')
print()
print('Score distribution (decile buckets):')
print(
    df_checkov['checkov_score']
    .pipe(lambda s: pd.cut(s, bins=10, precision=1))
    .value_counts(sort=False)
    .to_string()
)
print()
print('Top 5 sources by Checkov pass-rate:')
print(
    df_checkov.groupby('source_slug')['checkov_score']
    .mean().sort_values(ascending=False).head(5)
    .apply(lambda v: f'{v:.1%}')
    .to_string()
)
print()
print('Top 10 Checkov violations in corpus:')
all_violations = (
    df_checkov['checkov_top_violations']
    .dropna().str.split('|').explode()
    .loc[lambda s: s != '']
)
top10 = Counter(all_violations).most_common(10)
print(f"{'Check ID':<25} {'Count':>6}")
print('-' * 33)
for check_id, count in top10:
    print(f'{check_id:<25} {count:>6,}')


✅ Loaded Checkov cache (30,534 entries)


Checkov scan:   0%|          | 0/30600 [00:00<?, ?it/s]


✅ Checkov cache saved → dataset/checkov_cache.csv
CHECKOV SECURITY SCAN SUMMARY
  Input (AWS-filtered)          : 30,600
  ✅  Pass (score ≥ 50%)       : 25,536
  ❌  Fail (score < 50%)       :  5,064
  📊  Average pass-rate          :    80.5%

Score distribution (decile buckets):
checkov_score
(-0.001, 0.1]     1188
(0.1, 0.2]         403
(0.2, 0.3]        1197
(0.3, 0.4]        1529
(0.4, 0.5]        2082
(0.5, 0.6]        1507
(0.6, 0.7]        1212
(0.7, 0.8]        1212
(0.8, 0.9]         801
(0.9, 1.0]       19469

Top 5 sources by Checkov pass-rate:
source_slug
alfonsof/terraform-aws-examples            100.0%
terraform-aws-modules/terraform-aws-vpc    100.0%
aws-samples/cict-terraform-scripts          87.9%
ovotech/terraform-testing-samples           83.3%
Tianyi2/IRIS                                80.9%

Top 10 Checkov violations in corpus:
Check ID                   Count
---------------------------------
nan                       19,094
CKV_TF_1                   1,913
CKV_A

In [18]:
rc = subprocess.run(['which', 'trivy'], capture_output=True).returncode
status = '✅' if rc == 0 else '⚠️  NOT FOUND – install before running cells'
print(f'{'trivy':12s} {status}')

trivy        ✅


In [19]:
# ── 8c. Trivy IaC Security Scan ───────────────────────────────────────────────
import json, subprocess
from collections import Counter
from pathlib import Path

TRIVY_TIMEOUT        = 300
TRIVY_MIN_PASS       = 0.50
TRIVY_CACHE_CSV      = DATASET_DIR / 'trivy_cache.csv'
TRIVY_CACHE_FLUSH_EVERY = 50
TRIVY_MAX_VIOLATIONS = 10
TRIVY_SEVERITY_ORDER  = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'UNKNOWN']
TRIVY_SEVERITY_DEFAULT = 'NONE'

# ── Canonical column list (must match what run_trivy returns) ─────────────────
_TRIVY_COLS = [
    'trivy_passed', 'trivy_failed',
    'trivy_ppr',                        # Pass-rate  = passed / total
    'trivy_fcr',                        # Failure Count Ratio = failed / total
    'trivy_ok',
    'trivy_max_severity',
    'trivy_top_violations',
    # Per-severity failure counts
    'trivy_n_critical', 'trivy_n_high',
    'trivy_n_medium',   'trivy_n_low',
    'trivy_n_unknown',
]

_TRIVY_DEFAULTS = dict(
    trivy_passed=0, trivy_failed=0,
    trivy_ppr=0.0,  trivy_fcr=0.0,
    trivy_ok=False,
    trivy_max_severity=TRIVY_SEVERITY_DEFAULT,
    trivy_top_violations='',
    trivy_n_critical=0, trivy_n_high=0,
    trivy_n_medium=0,   trivy_n_low=0,
    trivy_n_unknown=0,
)

# ── Load or create cache ───────────────────────────────────────────────────────
if TRIVY_CACHE_CSV.exists():
    _tv_cache = pd.read_csv(TRIVY_CACHE_CSV, index_col='content_hash')
    # Restore dtypes — booleans are read back as strings
    _tv_cache['trivy_ok'] = _tv_cache['trivy_ok'].astype(str).str.lower() == 'true'
    for col in ['trivy_passed', 'trivy_failed',
                'trivy_n_critical', 'trivy_n_high',
                'trivy_n_medium', 'trivy_n_low', 'trivy_n_unknown']:
        if col in _tv_cache.columns:
            _tv_cache[col] = pd.to_numeric(_tv_cache[col], errors='coerce').fillna(0).astype(int)
    for col in ['trivy_ppr', 'trivy_fcr']:
        if col in _tv_cache.columns:
            _tv_cache[col] = pd.to_numeric(_tv_cache[col], errors='coerce').fillna(0.0)
    # Back-fill any columns added since the cache was written
    for col, default in _TRIVY_DEFAULTS.items():
        if col not in _tv_cache.columns:
            _tv_cache[col] = default
    print(f'✅ Loaded Trivy cache ({len(_tv_cache):,} entries) — cached hashes will be skipped')
else:
    _tv_cache = pd.DataFrame(columns=_TRIVY_COLS)
    _tv_cache.index.name = 'content_hash'
    print('ℹ️  No existing Trivy cache — starting fresh')


# ── Per-template scanner ───────────────────────────────────────────────────────
def run_trivy(row) -> dict:
    chash = str(row.get('content_hash', ''))

    # ── Cache hit ─────────────────────────────────────────────────────────────
    if chash and chash in _tv_cache.index:
        cached = _tv_cache.loc[chash]
        return {c: cached.get(c, _TRIVY_DEFAULTS[c]) for c in _TRIVY_COLS}

    dest_file = BASE_DIR / row['dest_file']
    scan_dir  = dest_file.parent

    if not dest_file.exists():
        result = dict(_TRIVY_DEFAULTS)
        if chash:
            _tv_cache.loc[chash] = result
        return result

    # ── Initialise accumulators ───────────────────────────────────────────────
    passed, failed    = 0, 0
    violation_ids     = []
    sev_counts        = Counter()
    max_severity      = TRIVY_SEVERITY_DEFAULT

    try:
        proc = subprocess.run(
            ['trivy', 'config', '--format', 'json', '--quiet', str(scan_dir)],
            capture_output=True, text=True, timeout=TRIVY_TIMEOUT,
        )
        raw = proc.stdout.strip()
        if not raw:
            raise ValueError('empty stdout')
        data = json.loads(raw)

        for scan_result in data.get('Results', []):
            summary = scan_result.get('MisconfSummary', {})
            passed += int(summary.get('Successes', 0))
            failed += int(summary.get('Failures',  0))

            for m in scan_result.get('Misconfigurations', []):
                if m.get('Status') == 'FAIL':
                    violation_ids.append(m.get('ID', ''))
                    sev = (m.get('Severity') or '').upper()
                    if sev in TRIVY_SEVERITY_ORDER:
                        sev_counts[sev] += 1

        # Max severity — walk in priority order
        for sev in TRIVY_SEVERITY_ORDER:
            if sev_counts.get(sev, 0) > 0:
                max_severity = sev
                break

    except subprocess.TimeoutExpired:
        tqdm.write(f'  ⏱️  Trivy timeout: {scan_dir.name}')
    except Exception as e:
        tqdm.write(f'  ⚠️  Trivy error: {scan_dir.name}: {type(e).__name__}: {e}')

    total = passed + failed
    ppr   = round(passed / total, 4) if total > 0 else 1.0
    fcr   = round(failed / total, 4) if total > 0 else 0.0
    ok    = ppr >= TRIVY_MIN_PASS

    top_violations = '|'.join(
        vid for vid, _ in Counter(violation_ids).most_common(TRIVY_MAX_VIOLATIONS)
    )

    result = dict(
        trivy_passed        = passed,
        trivy_failed        = failed,
        trivy_ppr           = ppr,
        trivy_fcr           = fcr,
        trivy_ok            = ok,
        trivy_max_severity  = max_severity,
        trivy_top_violations= top_violations,
        trivy_n_critical    = sev_counts.get('CRITICAL', 0),
        trivy_n_high        = sev_counts.get('HIGH',     0),
        trivy_n_medium      = sev_counts.get('MEDIUM',   0),
        trivy_n_low         = sev_counts.get('LOW',      0),
        trivy_n_unknown     = sev_counts.get('UNKNOWN',  0),
    )

    if chash:
        _tv_cache.loc[chash] = result

    return result


# ── Batch run with periodic cache flush ───────────────────────────────────────
# ── Batch run with periodic cache flush ───────────────────────────────────────
trivy_rows  = []
new_scanned = 0

for i, (_, row) in enumerate(tqdm(df_checkov.iterrows(),
                                   total=len(df_checkov), desc='Trivy scan')):
    chash      = str(row.get('content_hash', ''))
    was_cached = bool(chash and chash in _tv_cache.index)   # check BEFORE run_trivy

    trivy_rows.append(run_trivy(row))

    if not was_cached:                                        # only count actual new scans
        new_scanned += 1
        if new_scanned % TRIVY_CACHE_FLUSH_EVERY == 0:
            _tv_cache.to_csv(TRIVY_CACHE_CSV, index=True)
            tqdm.write(f'  💾 Trivy cache flushed ({new_scanned} new scans written)')

# Final flush
_tv_cache.to_csv(TRIVY_CACHE_CSV, index=True)
print(f'\n✅ Trivy cache saved → {TRIVY_CACHE_CSV.relative_to(BASE_DIR)}')

df_trivy_meta = pd.DataFrame(trivy_rows, index=df_checkov.index)
df_trivy      = df_checkov.join(df_trivy_meta)


# ── Summary ───────────────────────────────────────────────────────────────────
n_total   = len(df_trivy)
n_ok      = df_trivy['trivy_ok'].sum()
avg_ppr   = df_trivy['trivy_ppr'].mean()
avg_fcr   = df_trivy['trivy_fcr'].mean()

print('=' * 60)
print('TRIVY MISCONFIG SCAN SUMMARY')
print('=' * 60)
print(f'  Input (AWS-filtered)           : {n_total:>6,}')
print(f'  ✅  Pass (PPR ≥ {TRIVY_MIN_PASS:.0%})         : {n_ok:>6,}')
print(f'  ❌  Fail (PPR < {TRIVY_MIN_PASS:.0%})         : {n_total - n_ok:>6,}')
print(f'  📊  Mean PPR (pass-rate)        : {avg_ppr:>8.1%}')
print(f'  📊  Mean FCR (failure-rate)     : {avg_fcr:>8.1%}')
print()
print('Severity distribution (failing checks):')
sev_summary = (
    df_trivy['trivy_max_severity']
    .value_counts()
    .reindex([*TRIVY_SEVERITY_ORDER, TRIVY_SEVERITY_DEFAULT], fill_value=0)
)
print(sev_summary.to_string())
print()
print('Per-severity failure totals across all templates:')
for sev in TRIVY_SEVERITY_ORDER:
    col = f'trivy_n_{sev.lower()}'
    print(f'  {sev:<10}: {int(df_trivy[col].sum()):>8,}')
print()
print('Top 5 sources by Trivy PPR:')
print(
    df_trivy.groupby('source_slug')['trivy_ppr']
    .mean().sort_values(ascending=False).head(5)
    .apply(lambda v: f'{v:.1%}')
    .to_string()
)
print()
print('Top 10 most frequent violations:')
all_violations = Counter(
    vid
    for ids in df_trivy['trivy_top_violations'].dropna()
    for vid in ids.split('|') if vid
)
for vid, cnt in all_violations.most_common(10):
    print(f'  {vid:<30} {cnt:>6,}')

✅ Loaded Trivy cache (30,549 entries) — cached hashes will be skipped


Trivy scan:   0%|          | 0/30600 [00:00<?, ?it/s]


✅ Trivy cache saved → dataset/trivy_cache.csv
TRIVY MISCONFIG SCAN SUMMARY
  Input (AWS-filtered)           : 30,600
  ✅  Pass (PPR ≥ 50%)         : 30,054
  ❌  Fail (PPR < 50%)         :    546
  📊  Mean PPR (pass-rate)        :    96.2%
  📊  Mean FCR (failure-rate)     :     3.4%

Severity distribution (failing checks):
trivy_max_severity
CRITICAL     2399
HIGH         3293
MEDIUM       1766
LOW           602
UNKNOWN         0
NONE        22540

Per-severity failure totals across all templates:
  CRITICAL  :   44,107
  HIGH      :  282,094
  MEDIUM    :  177,391
  LOW       :  163,478
  UNKNOWN   :        0

Top 5 sources by Trivy PPR:
source_slug
alfonsof/terraform-aws-examples            100.0%
terraform-aws-modules/terraform-aws-vpc    100.0%
ovotech/terraform-testing-samples           99.1%
Tianyi2/IRIS                                97.4%
aws-samples/cict-terraform-scripts          96.1%

Top 10 most frequent violations:
  AWS-0178                        1,600
  AWS-0124       

In [20]:
# ── 8d. Security Threshold Filter (PPR + severity guardrail) ──────────────────
# Input  : df_trivy  (df_aws + checkov + trivy outputs)
# Output : df_secure filtered by score thresholds and severity inclusion policy

CHECKOV_PPR_THRESHOLD = 0.0
TRIVY_PPR_THRESHOLD = 1.0
EXCLUDED_TRIVY_SEVERITIES = {'NONE', 'UNKNOWN'}

mask_checkov = df_trivy['checkov_score'] >= CHECKOV_PPR_THRESHOLD
mask_trivy   = df_trivy['trivy_ppr']   >= TRIVY_PPR_THRESHOLD
sev_series   = df_trivy['trivy_max_severity'].fillna('UNKNOWN').astype(str).str.upper()
mask_sev     = ~sev_series.isin(EXCLUDED_TRIVY_SEVERITIES)

# df_secure = df_trivy[mask_checkov & mask_trivy & mask_sev].copy()
df_secure = df_trivy[mask_checkov & mask_trivy].copy()

# ── Persist security-filtered index for the deploy script ─────────────────────
_secure_index_csv = BASE_DIR / 'dataset' / 'security_filtered_index.csv'
df_secure.to_csv(_secure_index_csv, index=False)
print(f'✅ Security-filtered index → {_secure_index_csv.relative_to(BASE_DIR)}')

# ── Summary ───────────────────────────────────────────────────────────────────
n_in            = len(df_trivy)
n_checkov_pass  = int(mask_checkov.sum())
n_trivy_pass    = int(mask_trivy.sum())
# n_severity_keep = int(mask_sev.sum())
n_both_pass     = len(df_secure)

print()
print('=' * 60)
print('SECURITY FILTER')
print('=' * 60)
print(f'  Input (AWS-targeting)         : {n_in:>6,}')
print(f'  ✅  Checkov ≥ {CHECKOV_PPR_THRESHOLD:.0%}             : {n_checkov_pass:>6,}')
print(f'  ✅  Trivy   ≥ {TRIVY_PPR_THRESHOLD:.0%}             : {n_trivy_pass:>6,}')
# print(f'  ✅  Severity allowed           : {n_severity_keep:>6,} (exclude {sorted(EXCLUDED_TRIVY_SEVERITIES)})')
print(f'  ✅  Final df_secure            : {n_both_pass:>6,}  ({n_both_pass/n_in:.1%})')
print(f'  ❌  Dropped total              : {n_in - n_both_pass:>6,}')
print()
print('Severity distribution after filter:')
print(df_secure['trivy_max_severity'].fillna('UNKNOWN').value_counts().to_string())
df_secure.head()

✅ Security-filtered index → dataset/security_filtered_index.csv

SECURITY FILTER
  Input (AWS-targeting)         : 30,600
  ✅  Checkov ≥ 0%             : 30,600
  ✅  Trivy   ≥ 100%             : 22,540
  ✅  Final df_secure            : 22,540  (73.7%)
  ❌  Dropped total              :  8,060

Severity distribution after filter:
trivy_max_severity
NONE    22540


,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,...,trivy_ppr,trivy_fcr,trivy_ok,trivy_max_severity,trivy_top_violations,trivy_n_critical,trivy_n_high,trivy_n_medium,trivy_n_low,trivy_n_unknown
12,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,1.0,0.0,True,NONE,NaN,0,0,0,0,0
13,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,1.0,0.0,True,NONE,NaN,0,0,0,0,0
15,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,5,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,1.0,0.0,True,NONE,NaN,0,0,0,0,0
16,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,1.0,0.0,True,NONE,NaN,0,0,0,0,0
20,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,1.0,0.0,True,NONE,NaN,0,0,0,0,0


In [21]:
# ── 8e. Structural features: n_resources, n_params, difficulty ───────────────
# Extracts per-file structural signals used for the DPIaC-style composite score.
#   n_resources – number of Terraform resource blocks
#   n_params    – number of variable blocks (proxy for configuration parameters)
#   difficulty  – LoC-based bucket (1–5) matching IaC-Eval / DPIaC-Eval lineage

import re, io
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

# ── LoC-based difficulty bucket ───────────────────────────────────────────────
def assign_difficulty(loc: int) -> int:
    if loc < 50:  return 1
    if loc < 100: return 2
    if loc < 150: return 3
    if loc < 200: return 4
    return 5

df_secure["difficulty"] = df_secure["loc"].apply(assign_difficulty)


# ── HCL2 resource + variable block counts ────────────────────────────────────
def count_hcl2_blocks(content: str) -> tuple[int, int]:
    """Return (n_resources, n_variables) for a single .tf file."""
    try:
        import hcl2
        tree = hcl2.load(io.StringIO(content))
        n_res = sum(len(v) for k, v in tree.items() if k == "resource")
        n_var = sum(len(v) for k, v in tree.items() if k == "variable")
        return n_res, n_var
    except Exception:
        # Regex fallback – should be rare since all rows passed hcl2_valid
        n_res = len(re.findall(r'^\s*resource\s+"', content, re.MULTILINE))
        n_var = len(re.findall(r'^\s*variable\s+"', content, re.MULTILINE))
        return n_res, n_var

def count_root_blocks(row) -> tuple[int, int]:
    """Count (resources, variables) across all .tf files in the template root."""
    dest_rel = str(row.get("dest_file", "") or "").strip()

    # Preferred: count from the on-disk root path used by upstream cells
    if dest_rel:
        root_dir = (BASE_DIR / dest_rel).parent
        if root_dir.exists() and root_dir.is_dir():
            n_res_total, n_var_total = 0, 0
            tf_files = sorted(root_dir.glob('*.tf'))
            for tf in tf_files:
                try:
                    text = tf.read_text(encoding='utf-8', errors='replace')
                except Exception:
                    continue
                n_res, n_var = count_hcl2_blocks(text)
                n_res_total += n_res
                n_var_total += n_var
            if tf_files:
                return n_res_total, n_var_total

    # Fallback: keep prior behaviour when dest_file is missing
    return count_hcl2_blocks(str(row.get("content", "") or ""))

import pandas as pd
import matplotlib.pyplot as plt

# 1. Calculate Resources and Variables
print("Counting resources and variables in df_secure root paths…")
_counts = df_secure.apply(count_root_blocks, axis=1)
df_secure["n_resources"] = _counts.apply(lambda t: t[0])
df_secure["n_params"]    = _counts.apply(lambda t: t[1])

# 2. Extract Summary Statistics
LOC_RANGES = {1: "< 50", 2: "50–99", 3: "100–149", 4: "150–199", 5: "≥ 200"}
total = len(df_secure)

stats = []
for lvl in range(1, 6):
    sub = df_secure[df_secure["difficulty"] == lvl]
    cnt = len(sub)
    pct = (cnt / total * 100) if total > 0 else 0
    stats.append({
        "Level": str(lvl),
        "LoC range": LOC_RANGES.get(lvl, ""),
        "Count": cnt,
        "%": pct,
        "Median LOC": sub['loc'].median() if cnt > 0 else 0,
        "Median res": sub['n_resources'].median() if cnt > 0 else 0,
        "Median vars": sub['n_params'].median() if cnt > 0 else 0,
    })

df_stats = pd.DataFrame(stats)

# 3. Create the Figure Plot
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle(f"Structural Features by Difficulty Level (n={total:,})", fontsize=16, fontweight='bold')

# Colors for different metrics
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B3']

# Panel 1: Distribution (Count & Percentage)
ax1 = axes[0, 0]
bars = ax1.bar(df_stats["Level"], df_stats["Count"], color=colors[0], edgecolor='black')
ax1.set_title("Distribution of Difficulty Levels", fontsize=12)
ax1.set_xlabel("Difficulty Level (1-5)")
ax1.set_ylabel("Count")
ax1.set_ylim(0, df_stats["Count"].max() * 1.15) # Add headroom for labels

# Add percentage labels on top of bars
for bar, pct in zip(bars, df_stats["%"]):
    height = bar.get_height()
    ax1.annotate(f"{pct:.1f}%",
                 xy=(bar.get_x() + bar.get_width() / 2, height),
                 xytext=(0, 4), textcoords="offset points",
                 ha='center', va='bottom', fontsize=10)

# Panel 2: Median LOC
ax2 = axes[0, 1]
ax2.bar(df_stats["Level"], df_stats["Median LOC"], color=colors[1], edgecolor='black')
ax2.set_title("Median Lines of Code (LOC)", fontsize=12)
ax2.set_xlabel("Difficulty Level")
ax2.set_ylabel("Median LOC")

# Panel 3: Median Resources
ax3 = axes[1, 0]
ax3.bar(df_stats["Level"], df_stats["Median res"], color=colors[2], edgecolor='black')
ax3.set_title("Median Resources", fontsize=12)
ax3.set_xlabel("Difficulty Level")
ax3.set_ylabel("Median Count")

# Panel 4: Median Variables/Params
ax4 = axes[1, 1]
ax4.bar(df_stats["Level"], df_stats["Median vars"], color=colors[3], edgecolor='black')
ax4.set_title("Median Variables/Params", fontsize=12)
ax4.set_xlabel("Difficulty Level")
ax4.set_ylabel("Median Count")

# Format and Save
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
save_path = "structural_features_summary.png"
plt.savefig(save_path, dpi=300, bbox_inches='tight')
plt.close()

# 4. Print Global Summaries to Console
print(f"Figure successfully saved to: {save_path}\n")
print(f"  Total rows     : {total:,}")
print(f"  Median LOC     : {df_secure['loc'].median():.0f}")
print(f"  Median res     : {df_secure['n_resources'].median():.1f}")
print(f"  Median vars    : {df_secure['n_params'].median():.1f}")

Counting resources and variables in df_secure root paths…
Figure successfully saved to: structural_features_summary.png

  Total rows     : 22,540
  Median LOC     : 27
  Median res     : 1.0
  Median vars    : 0.0


In [22]:
# ── 8f-pre. Empirical threshold analysis (LoC, resources, params, security) ─────
import numpy as np
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

def _midpoint_thresholds_from_medians(series: pd.Series, by_level: pd.Series, fallback: list[int]) -> list[int]:
    """Empirical thresholds via midpoints between adjacent per-level medians."""
    med = (
        pd.DataFrame({'v': pd.to_numeric(series, errors='coerce'), 'lvl': by_level})
        .dropna()
        .groupby('lvl')['v']
        .median()
        .reindex([1, 2, 3, 4, 5])
    )
    if med.isna().any():
        return fallback

    vals = []
    for i in range(1, 5):
        mid = (float(med.loc[i]) + float(med.loc[i + 1])) / 2.0
        vals.append(max(1, int(round(mid))))

    # Ensure strictly increasing cutoffs
    for i in range(1, 4):
        if vals[i] <= vals[i - 1]:
            vals[i] = vals[i - 1] + 1

    return vals

SECURITY_ORDINAL = ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']

# LoC threshold is intentionally fixed for interpretability
LOC_THRESH_EMP = [50, 100, 150, 200]

# Structural thresholds are empirical from observed medians by current difficulty band
RES_THRESH_EMP = _midpoint_thresholds_from_medians(
    df_secure['n_resources'],
    df_secure['difficulty'],
    fallback=[2, 4, 6, 8],
)
VAR_THRESH_EMP = _midpoint_thresholds_from_medians(
    df_secure['n_params'],
    df_secure['difficulty'],
    fallback=[2, 4, 6, 8],
)

# Security is kept as ordinal categorical data (no numeric threshold fitting)
sev_series = df_secure['trivy_max_severity'].fillna('UNKNOWN').astype(str).str.upper()

tbl = Table(title='Empirical Thresholds for Composite Difficulty', box=box.ROUNDED, show_lines=True)
tbl.add_column('Dimension', justify='left', style='bold')
tbl.add_column('Method', justify='left')
tbl.add_column('Thresholds (L1|L2|L3|L4)', justify='left')
tbl.add_row('LoC', 'Fixed interpretable cutoffs', str(LOC_THRESH_EMP))
tbl.add_row('Resources', 'Midpoints of per-level medians', str(RES_THRESH_EMP))
tbl.add_row('Params', 'Midpoints of per-level medians', str(VAR_THRESH_EMP))
tbl.add_row('Trivy severity', 'Ordinal categories used directly', 'LOW < MEDIUM < HIGH < CRITICAL')

console.print()
console.print(tbl)
n_ordinal = int(sev_series.isin(SECURITY_ORDINAL).sum())
console.print(f"Security ordinal coverage: {n_ordinal:,} rows")

console.print('Severity distribution (trivy_max_severity):')
console.print(sev_series.value_counts())

df_secure.drop(columns=['_trivy_sev_score_for_thresholds'], inplace=True, errors='ignore')

                    Empirical Thresholds for Composite Difficulty                     
╭────────────────┬──────────────────────────────────┬────────────────────────────────╮
│ Dimension      │ Method                           │ Thresholds (L1|L2|L3|L4)       │
├────────────────┼──────────────────────────────────┼────────────────────────────────┤
│ LoC            │ Fixed interpretable cutoffs      │ [50, 100, 150, 200]            │
├────────────────┼──────────────────────────────────┼────────────────────────────────┤
│ Resources      │ Midpoints of per-level medians   │ [2, 4, 5, 6]                   │
├────────────────┼──────────────────────────────────┼────────────────────────────────┤
│ Params         │ Midpoints of per-level medians   │ [1, 4, 6, 8]                   │
├────────────────┼──────────────────────────────────┼────────────────────────────────┤
│ Trivy severity │ Ordinal categories used directly │ LOW < MEDIUM < HIGH < CRITICAL │
╰────────────────┴──────────────────────────────────┴────────────────────────────────╯

Security ordinal coverage: 0 rows

Severity distribution (trivy_max_severity):

trivy_max_severity
NONE    22540
Name: count, dtype: int64

In [23]:
# ── 8f. Composite difficulty score (fully threshold-based) ─────────────────────
# Dimensions used: LoC, n_resources, n_params (security severity ignored for now)

import numpy as np
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

# Pull empirically-derived thresholds from previous cell (with safe defaults)
LOC_THRESH = globals().get('LOC_THRESH_EMP', [50, 100, 150, 200])
RES_THRESH = globals().get('RES_THRESH_EMP', [2, 4, 6, 8])
VAR_THRESH = globals().get('VAR_THRESH_EMP', [2, 4, 6, 8])

def _sublevel(val, thresholds) -> int:
    if pd.isna(val):
        return 1
    for lvl, thr in enumerate(thresholds, start=1):
        if float(val) < float(thr):
            return lvl
    return 5

# Structural sublevels by dimension
df_secure['loc_level'] = df_secure['loc'].apply(lambda x: _sublevel(x, LOC_THRESH))
df_secure['res_level'] = df_secure['n_resources'].apply(lambda x: _sublevel(x, RES_THRESH))
df_secure['par_level'] = df_secure['n_params'].apply(lambda x: _sublevel(x, VAR_THRESH))

# Keep DPIaC-style structural level (max of structural dimensions) for reference
df_secure['dpiac_level'] = df_secure[['loc_level', 'res_level', 'par_level']].max(axis=1)

# Composite score: weighted average of structural dimensions only
W_LOC = 0.4
W_RES = 0.3
W_PAR = 0.3

df_secure['composite_score'] = (
    W_LOC * df_secure['loc_level'] +
    W_RES * df_secure['res_level'] +
    W_PAR * df_secure['par_level']
).round(3)

# Composite level is also threshold-based (fixed cutoffs on 1–5 score)
COMPOSITE_THRESH = [1.5, 2.5, 3.5, 4.5]
df_secure['composite_level'] = df_secure['composite_score'].apply(
    lambda s: _sublevel(s, COMPOSITE_THRESH)
)

df_secure.drop(columns=['_trivy_sev_score'], inplace=True, errors='ignore')

console.print("\n[bold]Structural level (DPIaC-style):[/bold]")
console.print(df_secure['dpiac_level'].value_counts().sort_index())
console.print(f"Dimension thresholds → LOC: {LOC_THRESH}, resources: {RES_THRESH}, params: {VAR_THRESH}")

tbl = Table(title='Composite Difficulty Levels (Threshold-Based)', box=box.ROUNDED, show_lines=True)
for col in ['Level', 'Count', '%', 'Median LOC', 'Median resources', 'Median params']:
    tbl.add_column(col, justify='right' if col != 'Level' else 'center')

for level in range(1, 6):
    sub = df_secure[df_secure['composite_level'] == level]
    n = len(sub)
    tbl.add_row(
        str(level),
        str(n),
        f"{(n / len(df_secure) * 100):.1f}%",
        f"{sub['loc'].median():.0f}" if n else 'N/A',
        f"{sub['n_resources'].median():.0f}" if n else 'N/A',
        f"{sub['n_params'].median():.0f}" if n else 'N/A',
        # f"{sub['checkov_score'].median():.2f}" if n and sub['checkov_score'].notna().any() else 'N/A',
        # str(sub['trivy_max_severity'].mode().iloc[0]) if n and sub['trivy_max_severity'].notna().any() else 'N/A',
    )

console.print()
console.print(tbl)
console.print(f"\n[bold]Composite score range:[/bold] {df_secure['composite_score'].min():.3f} – {df_secure['composite_score'].max():.3f}")
console.print(f"[bold]Composite thresholds:[/bold] {COMPOSITE_THRESH}")

out_path = DATASET_DIR / 'df_secure_with_difficulty.csv'
df_secure.to_csv(out_path, index=False)
console.print(f"\n[green]✅ Saved:[/green] {out_path}")
console.print('   New cols: loc_level, res_level, par_level, dpiac_level, composite_score, composite_level')

Structural level (DPIaC-style):

dpiac_level
1    5830
2    6721
3    2701
4    1866
5    5422
Name: count, dtype: int64

Dimension thresholds → LOC: [50, 100, 150, 200], resources: [2, 4, 5, 6], params: [1, 4, 6, 8]

              Composite Difficulty Levels (Threshold-Based)              
╭───────┬───────┬───────┬────────────┬──────────────────┬───────────────╮
│ Level │ Count │     % │ Median LOC │ Median resources │ Median params │
├───────┼───────┼───────┼────────────┼──────────────────┼───────────────┤
│   1   │ 10595 │ 47.0% │         16 │                1 │             0 │
├───────┼───────┼───────┼────────────┼──────────────────┼───────────────┤
│   2   │  6928 │ 30.7% │         34 │                2 │             3 │
├───────┼───────┼───────┼────────────┼──────────────────┼───────────────┤
│   3   │  3160 │ 14.0% │         77 │                4 │             7 │
├───────┼───────┼───────┼────────────┼──────────────────┼───────────────┤
│   4   │  1372 │  6.1% │        140 │                6 │            10 │
├───────┼───────┼───────┼────────────┼──────────────────┼───────────────┤
│   5   │   485 │  2.2% │        236 │               10 │            19 │
╰───────┴───────┴───────┴────────────┴──────────────────┴───────────────╯

Composite score range: 1.000 – 5.000

Composite thresholds: [1.5, 2.5, 3.5, 4.5]

✅ Saved: iac_benchmark/dataset/df_secure_with_difficulty.csv

New cols: loc_level, res_level, par_level, dpiac_level, composite_score, composite_level

In [24]:
df_secure.head()

,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,...,trivy_n_low,trivy_n_unknown,n_resources,n_params,loc_level,res_level,par_level,dpiac_level,composite_score,composite_level
12,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,1,16,3,1,5,5,3.0,3
13,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,6,27,5,5,5,5,5.0,5
15,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,5,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,5,23,4,4,5,5,4.3,4
16,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,8,36,5,5,5,5,5.0,5
20,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,2,0,2,2,1,2,1.7,2


In [25]:
# ── 8g. Balanced sample: 200 files evenly across 5 composite levels ─────────────
import pandas as pd
from rich.console import Console
from rich.table import Table
from rich import box

console = Console()

TARGET_TOTAL = 12000
LEVELS = [1, 2, 3, 4, 5]
RANDOM_SEED = 42

if 'df_secure' not in globals() or 'composite_level' not in df_secure.columns:
    raise RuntimeError("Run the composite difficulty cell first so df_secure['composite_level'] exists.")

per_level = TARGET_TOTAL // len(LEVELS)
if per_level * len(LEVELS) != TARGET_TOTAL:
    raise ValueError('TARGET_TOTAL must be divisible by 5 for an even split.')

parts = []
sampling_meta = []

for lvl in LEVELS:
    sub = df_secure[df_secure['composite_level'] == lvl].copy()
    available = len(sub)
    replace = available < per_level

    sampled = sub.sample(
        n=per_level,
        replace=replace,
        random_state=RANDOM_SEED + lvl,
    )
    sampled['balanced_sample_level'] = lvl
    parts.append(sampled)

    sampling_meta.append({
        'level': lvl,
        'available': available,
        'sampled': per_level,
        'replace': replace,
    })

df_balanced_200 = pd.concat(parts, ignore_index=True)
df_balanced_200 = df_balanced_200.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

# Persist a reproducible balanced subset for downstream use
out_path_balanced = DATASET_DIR / 'df_secure_balanced_200.csv'
df_balanced_200.to_csv(out_path_balanced, index=False)

tbl = Table(title='Balanced 200-file Sample by Composite Level', box=box.ROUNDED, show_lines=True)
tbl.add_column('Level', justify='center')
tbl.add_column('Available', justify='right')
tbl.add_column('Sampled', justify='right')
tbl.add_column('With replacement', justify='center')

for rec in sampling_meta:
    tbl.add_row(
        str(rec['level']),
        f"{rec['available']:,}",
        f"{rec['sampled']:,}",
        'Yes' if rec['replace'] else 'No',
    )

console.print(tbl)
console.print(f"\nTotal sampled rows: {len(df_balanced_200):,}")
console.print(f"Saved: {out_path_balanced}")
console.print("\nDistribution in sampled set:")
console.print(df_balanced_200['composite_level'].value_counts().sort_index())

df_balanced_200.head()

   Balanced 200-file Sample by Composite Level    
╭───────┬───────────┬─────────┬──────────────────╮
│ Level │ Available │ Sampled │ With replacement │
├───────┼───────────┼─────────┼──────────────────┤
│   1   │    10,595 │   2,400 │        No        │
├───────┼───────────┼─────────┼──────────────────┤
│   2   │     6,928 │   2,400 │        No        │
├───────┼───────────┼─────────┼──────────────────┤
│   3   │     3,160 │   2,400 │        No        │
├───────┼───────────┼─────────┼──────────────────┤
│   4   │     1,372 │   2,400 │       Yes        │
├───────┼───────────┼─────────┼──────────────────┤
│   5   │       485 │   2,400 │       Yes        │
╰───────┴───────────┴─────────┴──────────────────╯

Total sampled rows: 12,000

Saved: iac_benchmark/dataset/df_secure_balanced_200.csv

Distribution in sampled set:

composite_level
1    2400
2    2400
3    2400
4    2400
5    2400
Name: count, dtype: int64

,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,...,trivy_n_unknown,n_resources,n_params,loc_level,res_level,par_level,dpiac_level,composite_score,composite_level,balanced_sample_level
0,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_3...,1,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,1,0,1,1,1,1,1.0,1,1
1,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_2...,5,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,12,5,1,5,3,5,2.8,3,3
2,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_2...,1,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,2,0,1,2,1,2,1.3,1,1
3,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_1...,1,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,3,39,5,2,5,5,4.1,4,4
4,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_1...,1,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,1,3,1,1,2,2,1.3,1,1


In [28]:
# ── 8g. Deployable-only analysis + final benchmark stratification ──────────────
from pathlib import Path
import numpy as np
import pandas as pd


# Resolve dataset directory from prior notebook context when available
if 'DATASET_DIR' in globals():
    dataset_dir = Path(DATASET_DIR)
elif 'BASE_DIR' in globals():
    dataset_dir = Path(BASE_DIR) / 'dataset'
else:
    dataset_dir = Path('iac_benchmark/dataset')

DEPLOY_CSV = dataset_dir / 'deployability_results_v5.csv'
SECURE_CSV = dataset_dir / 'df_secure_with_difficulty.csv'
OUT_STRATIFIED = dataset_dir / 'final_benchmark_deployable_stratified.csv'
OUT_SUMMARY = dataset_dir / 'final_benchmark_deployable_stratified_summary.csv'

if not DEPLOY_CSV.exists():
    raise FileNotFoundError(f'Missing deployability file: {DEPLOY_CSV}')
if not SECURE_CSV.exists():
    raise FileNotFoundError(f'Missing secure feature file: {SECURE_CSV}')

def to_bool(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.strip()
         .str.lower()
         .map({'true': True, 'false': False, '1': True, '0': False, 'yes': True, 'no': False})
    )

# 1) Read deployability results and retain deployable scenarios only
df_deploy = pd.read_csv(DEPLOY_CSV, low_memory=False)
df_deploy['apply_ok'] = to_bool(df_deploy['apply_ok'])

df_deploy_ok = df_deploy[df_deploy['apply_ok'] == True].copy()
if df_deploy_ok.empty:
    raise RuntimeError('No deployable scenarios found (apply_ok=True).')

# 2) Bring structural + security signals from secure dataset
need_cols = ['dest_file', 'n_resources', 'n_params', 'trivy_ppr', 'checkov_score', 'composite_level']
df_secure_features = pd.read_csv(
    SECURE_CSV,
    usecols=lambda c: c in set(need_cols),
    low_memory=False,
)

df_final_benchmark = df_deploy_ok.merge(
    df_secure_features,
    on='dest_file',
    how='left',
)

# Deduplicate if the deployability file contains repeated checks per template
df_final_benchmark = (
    df_final_benchmark
    .sort_values('checked_at' if 'checked_at' in df_final_benchmark.columns else 'dest_file')
    .drop_duplicates(subset=['dest_file'], keep='last')
    .reset_index(drop=True)
)

# 3) Numeric cleanup + analysis metrics
for col in ['loc', 'n_resources', 'n_params', 'trivy_ppr', 'checkov_score']:
    if col in df_final_benchmark.columns:
        df_final_benchmark[col] = pd.to_numeric(df_final_benchmark[col], errors='coerce')

# 4) Difficulty stratification based ONLY on LoC
# Fixed LoC cutoffs define the difficulty levels for all scenarios.
LOC_THRESH = globals().get('LOC_THRESH_EMP', [50, 100, 150, 200])
loc_bins = [-np.inf] + LOC_THRESH + [np.inf]
loc_labels = [1, 2, 3, 4, 5]

# Fill missing LoC values with the median so every deployable scenario gets a label.
loc_for_band = df_final_benchmark['loc'].fillna(df_final_benchmark['loc'].median())
df_final_benchmark['final_difficulty_score'] = loc_for_band
df_final_benchmark['final_difficulty_level'] = pd.cut(
    loc_for_band,
    bins=loc_bins,
    labels=loc_labels,
    include_lowest=True,
).astype(int)

band_map = {
    1: 'Very Easy',
    2: 'Easy',
    3: 'Moderate',
    4: 'Hard',
    5: 'Very Hard',
}
df_final_benchmark['final_difficulty_band'] = df_final_benchmark['final_difficulty_level'].map(band_map)

# 5) Summary table for benchmark design
summary = (
    df_final_benchmark
    .groupby('final_difficulty_level', as_index=False)
    .agg(
        n_templates=('dest_file', 'count'),
        median_loc=('loc', 'median'),
        min_loc=('loc', 'min'),
        max_loc=('loc', 'max'),
        median_resources=('n_resources', 'median'),
        median_params=('n_params', 'median'),
        min_score=('final_difficulty_score', 'min'),
        max_score=('final_difficulty_score', 'max'),
    )
)
summary['difficulty_band'] = summary['final_difficulty_level'].map(band_map)
summary = summary[['final_difficulty_level', 'difficulty_band', 'n_templates',
                   'median_loc', 'min_loc', 'max_loc', 'median_resources', 'median_params',
                   'min_score', 'max_score']]

# 6) Persist outputs used by final benchmark assembly
df_final_benchmark.to_csv(OUT_STRATIFIED, index=False)
summary.to_csv(OUT_SUMMARY, index=False)

print('=' * 88)
print('DEPLOYABLE SCENARIOS ANALYSIS (apply_ok=True)')
print('Difficulty stratified by fixed LoC thresholds')
print('=' * 88)
print(f'Input deployability file : {DEPLOY_CSV}')
print(f'Deployable templates     : {len(df_final_benchmark):,}')
print(f'LoC thresholds           : {LOC_THRESH}')
print('Difficulty bands          : 1<=50, 2=51-100, 3=101-150, 4=151-200, 5>200')
print(f'Median LOC               : {df_final_benchmark["loc"].median():.1f}')
print(f'LOC range                : {df_final_benchmark["loc"].min():.0f} - {df_final_benchmark["loc"].max():.0f}')
print(f'Median #resources        : {df_final_benchmark["n_resources"].median():.1f}')
print(f'Median #params           : {df_final_benchmark["n_params"].median():.1f}')
print()
print('Difficulty distribution (threshold-based stratification):')
print(
    df_final_benchmark['final_difficulty_level']
    .value_counts()
    .sort_index()
    .to_string()
)
print()
print('Per-level benchmark summary (threshold-based difficulty):')
print(summary.to_string(index=False))
print()
print(f'Saved stratified dataset : {OUT_STRATIFIED}')
print(f'Saved level summary      : {OUT_SUMMARY}')

summary


DEPLOYABLE SCENARIOS ANALYSIS (apply_ok=True)
Difficulty stratified by fixed LoC thresholds
Input deployability file : iac_benchmark/dataset/deployability_results_v5.csv
Deployable templates     : 229
LoC thresholds           : [50, 100, 150, 200]
Difficulty bands          : 1<=50, 2=51-100, 3=101-150, 4=151-200, 5>200
Median LOC               : 31.0
LOC range                : 5 - 396
Median #resources        : 1.0
Median #params           : 2.0

Difficulty distribution (threshold-based stratification):
final_difficulty_level
1    141
2     31
3     18
4     18
5     21

Per-level benchmark summary (threshold-based difficulty):
 final_difficulty_level difficulty_band  n_templates  median_loc  min_loc  max_loc  median_resources  median_params  min_score  max_score
                      1       Very Easy          141        17.0        5       50               1.0            0.0          5         50
                      2            Easy           31        69.0       53      100      

,final_difficulty_level,difficulty_band,n_templates,median_loc,min_loc,max_loc,median_resources,median_params,min_score,max_score
0,1,Very Easy,141,17.0,5,50,1.0,0.0,5,50
1,2,Easy,31,69.0,53,100,2.0,10.0,53,100
2,3,Moderate,18,132.5,101,147,3.5,23.0,101,147
3,4,Hard,18,183.0,165,196,4.0,28.5,165,196
4,5,Very Hard,21,239.0,201,396,5.0,36.0,201,396


In [27]:
print(df_secure.columns)
df_secure.head()

Index(['source_slug', 'source_category', 'primary_cloud', 'licence_spdx',
       'root_path', 'origin_files', 'file_count', 'is_merged', 'dest_file',
       'github_url', 'content', 'content_hash', 'loc', 'tokens',
       'iac_eval_resource', 'iac_eval_prompt', 'iac_eval_intent',
       'iac_eval_rego', 'iac_eval_level', 'difficulty', 'hcl2_valid',
       'hcl2_error', 'tflint_pass', 'tflint_error', 'hcl2_aws',
       'hcl2_aws_reason', 'tflint_aws', 'tflint_aws_reason', 'targets_aws',
       'checkov_passed', 'checkov_failed', 'checkov_skipped', 'checkov_score',
       'checkov_ok', 'checkov_top_violations', 'checkov_violation_resources',
       'trivy_passed', 'trivy_failed', 'trivy_ppr', 'trivy_fcr', 'trivy_ok',
       'trivy_max_severity', 'trivy_top_violations', 'trivy_n_critical',
       'trivy_n_high', 'trivy_n_medium', 'trivy_n_low', 'trivy_n_unknown',
       'n_resources', 'n_params', 'loc_level', 'res_level', 'par_level',
       'dpiac_level', 'composite_score', 'composite_le

,source_slug,source_category,primary_cloud,licence_spdx,root_path,origin_files,file_count,is_merged,dest_file,github_url,...,trivy_n_low,trivy_n_unknown,n_resources,n_params,loc_level,res_level,par_level,dpiac_level,composite_score,composite_level
12,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,1,16,3,1,5,5,3.0,3
13,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,6,27,5,5,5,5,5.0,5
15,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,5,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,5,23,4,4,5,5,4.3,4
16,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,8,36,5,5,5,5,5.0,5
20,Tianyi2/IRIS,benchmark-source,multi,Apache-2.0,Tianyi2/IRIS/data/terraform_collected_template...,data/terraform_collected_templates_new/group_0...,4,False,multi_root_templates/Tianyi2__IRIS/data__terra...,https://github.com/Tianyi2/IRIS/tree/HEAD/data...,...,0,0,2,0,2,2,1,2,1.7,2


In [27]:
# ── 9. Deployability Check ────────────────────────────────────────────────────
# Deployability is handled by the EXTERNAL script:
#
#   python scripts/run_deployability_check.py \
#       --index  iac_benchmark/dataset/security_filtered_index.csv \  # ← CHANGED
#       --output iac_benchmark/dataset/deployability_results.csv \
#       --region ap-southeast-2 \
#       --sample 100 \
#       --resume
#
# Run the script in a terminal BEFORE executing this cell.
# ─────────────────────────────────────────────────────────────────────────────

from pathlib import Path
import pandas as pd

DEPLOY_RESULTS_CSV = BASE_DIR / 'dataset' / 'deployability_results_v5.csv'

if not DEPLOY_RESULTS_CSV.exists():
    print('⚠️  Deploy results CSV not found.')
    print(f'   Expected: {DEPLOY_RESULTS_CSV.resolve()}')
    print()
    print('   Run the following in a terminal first:')
    print()
    print('   python scripts/run_deployability_check.py \\')
    print(f'       --index  {BASE_DIR}/dataset/security_filtered_index.csv \\')  # ← CHANGED
    print(f'       --output {DEPLOY_RESULTS_CSV} \\')
    print(f'       --region {AWS_REGION} \\')
    print('       --sample 100 \\')
    print('       --resume')
    print()
    print('   Falling back to security-filtered set (no deploy filter).')
    df_deployable = df_secure.copy()                # ← CHANGED (was df_syntaxok[targets_aws])
    df_deployable['init_ok']      = None
    df_deployable['validate_ok']  = None
    df_deployable['plan_ok']      = None
    df_deployable['deploy_error'] = None
    df_deployable['deploy_stage'] = None
else:
    cols = [
    "source_slug","source_category","primary_cloud","licence_spdx",
    "dest_file","github_url","loc","tokens","difficulty",
    "hcl2_valid","tflint_pass","yaml_valid","targets_aws",
    "init_ok","validate_ok","apply_ok",
    "deploy_error","deploy_stage","reset_ok","backend","checked_at",
    ]
    
    dtype = {
        "source_slug":     "string",
        "source_category": "string",
        "primary_cloud":   "string",
        "licence_spdx":    "string",
        "dest_file":       "string",
        "github_url":      "string",
        "loc":             "Int64",   # nullable int
        "tokens":          "Int64",   # nullable int
        "difficulty":      "string",
        "hcl2_valid":      "boolean",
        "tflint_pass":     "boolean",
        "yaml_valid":      "boolean",
        "targets_aws":     "boolean",
        "init_ok":         "boolean",
        "validate_ok":     "boolean",
        "apply_ok":        "boolean",
        "deploy_error":    "string",
        "deploy_stage":    "string",
        "reset_ok":        "boolean",
        "backend":         "string",
        "checked_at":      "string",  # or "datetime64[ns]" after parse_dates
    }
    df_deploy = pd.read_csv(
        DEPLOY_RESULTS_CSV, 
        names=cols,
        header=0,
        dtype=dtype,
        on_bad_lines="skip",    # or "warn" to see the bad lines
        engine="python",        # more tolerant than C engine
    )

    merge_cols = ['dest_file', 'init_ok', 'validate_ok', 'apply_ok',
                  'deploy_error', 'deploy_stage', 'checked_at']

    df_deployable = (                               # ← CHANGED (was df_aws)
        df_secure                                   # ← CHANGED (was df_syntaxok[targets_aws])
        .merge(df_deploy[merge_cols], on='dest_file', how='left')
        .copy()
    )

    n_checked   = df_deployable['init_ok'].notna().sum()
    n_passed    = (df_deployable['apply_ok'] == True).sum()
    n_unchecked = df_deployable['init_ok'].isna().sum()

    print('=' * 60)
    print('DEPLOYABILITY RESULTS')
    print('=' * 60)
    print(f'  Security-filtered templates   : {len(df_deployable):>5,}')   # ← CHANGED label
    print(f'  Checked by script             : {n_checked:>5,}')
    print(f'  ✅  apply_ok (deployable)      : {n_passed:>5,}  ({100*n_passed/n_checked:.1f}% of checked)')
    print(f'  ⏭️   Not yet checked           : {n_unchecked:>5,}')
    print()

    if n_checked > 0:
        print('Failure stage breakdown:')
        failed_mask = df_deployable['apply_ok'] == False
        print(df_deployable.loc[failed_mask, 'deploy_stage'].value_counts().to_string())
        print()
        print('Pass rate by difficulty:')
        print(
            df_deployable[df_deployable['apply_ok'].notna()]
            .groupby('difficulty')['apply_ok']
            .agg(checked='count', passed='sum')
            .assign(pass_rate=lambda x: (x['passed'] / x['checked']).map('{:.1%}'.format))
            
            .to_string()
        )


DEPLOYABILITY RESULTS
  Security-filtered templates   : 22,540
  Checked by script             :     0
  ✅  apply_ok (deployable)      :     0  (nan% of checked)
  ⏭️   Not yet checked           : 22,540



/var/folders/mr/k47n2bks0fs1y0cmq922vqbh0000gn/T/ipykernel_41975/2649559752.py:98: RuntimeWarning: invalid value encountered in scalar divide
  print(f'  ✅  apply_ok (deployable)      : {n_passed:>5,}  ({100*n_passed/n_checked:.1f}% of checked)')


In [ ]:
# ── 10. Aggregate & Export ────────────────────────────────────────────────────
# Builds three output artefacts:
#   (a) full_dataset.csv         – all syntax-clean files across all clouds
#   (b) aws_syntax_clean.csv     – AWS-only, syntax validated
#   (c) aws_deployable.csv       – AWS-only, terraform plan passed (gold set)
#   (d) benchmark_summary.csv    – per-repository statistics

import pandas as pd
from tabulate import tabulate

# (a) Full syntax-clean dataset
df_full_export = df_syntaxok[[
    'source_slug','source_category','primary_cloud','licence_spdx',
    'dest_file','github_url','loc','tokens','difficulty',
    'hcl2_valid','tflint_pass','yaml_valid'
]].copy()
df_full_export.to_csv(DATASET_DIR / 'full_dataset.csv', index=False)

# (b) AWS syntax-clean
df_aws_export = df_aws[[
    'source_slug','source_category','primary_cloud','licence_spdx',
    'dest_file','github_url','loc','tokens','difficulty',
    'hcl2_valid','tflint_pass','yaml_valid'
]].copy()
df_aws_export.to_csv(DATASET_DIR / 'aws_syntax_clean.csv', index=False)

# (c) Deployable gold set (from sampled terraform plan)
if 'apply_ok' in df_deploy.columns and df_deploy['apply_ok'].notna().any():
    df_deployable = df_deploy[df_deploy['apply_ok'] == True][[
        'source_slug','source_category','primary_cloud','licence_spdx',
        'dest_file','github_url','loc','tokens','difficulty',
        'init_ok','validate_ok','apply_ok'
    ]].copy()
    df_deployable.to_csv(DATASET_DIR / 'aws_deployable.csv', index=False)
    gold_n = len(df_deployable)
else:
    gold_n = 0
    print('No deploy results – aws_deployable.csv skipped.')

# (d) Per-repo summary
summary = (
    df_syntaxok
    .groupby('source_slug')
    .agg(
        total_files  = ('dest_file', 'count'),
        mean_loc     = ('loc', 'mean'),
        mean_tokens  = ('tokens', 'mean'),
        licence      = ('licence_spdx', 'first'),
        cloud        = ('primary_cloud', 'first'),
        category     = ('source_category', 'first'),
    )
    .round(1)
    .reset_index()
    .sort_values('total_files', ascending=False)
)
summary.to_csv(DATASET_DIR / 'benchmark_summary.csv', index=False)

print('=' * 70)
print('BENCHMARK DATASET SUMMARY')
print('=' * 70)
print(f'  Total syntax-clean files : {len(df_syntaxok):>7,}')
print(f'  AWS-targeting files      : {len(df_aws):>7,}')
print(f'  Deployable (gold set)    : {gold_n:>7,}')
print(f'  Repositories included    : {len(summary):>7}')
print()
print(tabulate(summary, headers='keys', tablefmt='rounded_outline', showindex=False))
print()
print('Exports written to:', DATASET_DIR.resolve())


In [ ]:
# # ── 11. OPTIONAL – Checkov Security Scan (FIXED) ──────────────────────────────
# # Fixes:
# #   1. Checkov 3.x JSON schema change  (results is sometimes a list)
# #   2. ANSI / summary text prepended to stdout breaks json.loads
# #   3. Empty deployable set → ck_df has no columns → KeyError on access
# #
# # Reference metric – Filtered Compliance Rate (FCR):
# #   DPIaC-Eval baseline: 8.4%  (Zhang et al., 2025)

# import subprocess, json, re, tempfile, pathlib
# import pandas as pd
# from tqdm.notebook import tqdm

# CHECKOV_AVAILABLE = subprocess.run(
#     ['which', 'checkov'], capture_output=True
# ).returncode == 0

# # ── Robust JSON extractor ─────────────────────────────────────────────────────
# _JSON_RE = re.compile(r'(\{.*\}|\[.*\])', re.DOTALL)

# def _extract_json(raw: str):
#     """
#     Strip ANSI codes and any non-JSON prefix/suffix that Checkov 3.x prints.
#     Returns the parsed object or None.
#     """
#     # Remove ANSI escape codes
#     clean = re.sub(r'\x1b\[[0-9;]*m', '', raw)
#     # Try full string first (fast path)
#     try:
#         return json.loads(clean)
#     except json.JSONDecodeError:
#         pass
#     # Fallback: find the largest JSON block in stdout
#     match = _JSON_RE.search(clean)
#     if match:
#         try:
#             return json.loads(match.group(0))
#         except json.JSONDecodeError:
#             pass
#     return None

# # ── Normalise Checkov 2.x / 3.x schema ───────────────────────────────────────
# def _parse_checkov_output(obj) -> dict:
#     """
#     Checkov 2.x  → {"results": {"passed_checks": [...], "failed_checks": [...]}}
#     Checkov 3.x  → [{"check_id": ..., "passed": true/false, ...}, ...]
#                  OR {"results": {"passed_checks": [...], "failed_checks": [...]}}
#     """
#     if obj is None:
#         return {'checkov_passed': None, 'checkov_failed': None, 'checkov_fcr': None}

#     # ── Checkov 3.x list format ────────────────────────────────────────────────
#     if isinstance(obj, list):
#         passed = sum(1 for c in obj if c.get('passed') is True)
#         failed = sum(1 for c in obj if c.get('passed') is False)
#         fcr    = (failed == 0) if (passed + failed > 0) else None
#         return {'checkov_passed': passed, 'checkov_failed': failed, 'checkov_fcr': fcr}

#     # ── Checkov 2.x / 3.x dict format ─────────────────────────────────────────
#     if isinstance(obj, dict):
#         # Some 3.x builds wrap per-file results under a list at top level
#         if 'results' in obj:
#             results  = obj['results']
#         elif 'check_results' in obj:        # alternate key seen in some 3.x builds
#             results  = obj['check_results']
#         else:
#             results  = obj                  # treat the dict itself as results

#         # results may itself be a list of dicts
#         if isinstance(results, list):
#             return _parse_checkov_output(results)

#         passed = len(results.get('passed_checks', []))
#         failed = len(results.get('failed_checks', []))
#         fcr    = (failed == 0) if (passed + failed > 0) else None
#         return {'checkov_passed': passed, 'checkov_failed': failed, 'checkov_fcr': fcr}

#     return {'checkov_passed': None, 'checkov_failed': None, 'checkov_fcr': None}

# # ── Main scan function ────────────────────────────────────────────────────────
# def checkov_scan(content: str) -> dict:
#     with tempfile.TemporaryDirectory() as tmpd:
#         tf = pathlib.Path(tmpd) / 'main.tf'
#         tf.write_text(content, encoding='utf-8')
#         r = subprocess.run(
#             [
#                 'checkov', '-f', str(tf),
#                 '--framework', 'terraform',
#                 '-o', 'json',
#                 '--quiet',          # suppress progress bar
#                 '--compact',        # suppress passed-check details (smaller output)
#             ],
#             capture_output=True, text=True, timeout=90
#         )
#         # Checkov exits with code 1 when violations are found – that is expected.
#         # Only treat non-0/1 codes as hard errors.
#         if r.returncode not in (0, 1):
#             return {'checkov_passed': None, 'checkov_failed': None,
#                     'checkov_fcr': None, 'checkov_error': r.stderr[:300]}

#         obj = _extract_json(r.stdout)
#         result = _parse_checkov_output(obj)
#         result.setdefault('checkov_error', '')
#         return result

# # ── Guard: need actual plan results ──────────────────────────────────────────
# _has_plan = (
#     'plan_ok' in df_sample.columns
#     and df_sample['plan_ok'].notna().any()
# )

# if not CHECKOV_AVAILABLE:
#     print('⚠️  checkov not found on $PATH – skipping.\n'
#           '   Install with:  pip install checkov')

# elif not _has_plan:
#     print('⚠️  No terraform plan results found.\n'
#           '   Set SKIP_DEPLOY = False and re-run Cell 9 first.')

# else:
#     df_dep_for_checkov = df_sample[df_sample['plan_ok'] == True].copy()

#     if df_dep_for_checkov.empty:
#         print('⚠️  No deployable templates in the sample.\n'
#               '   Increase DEPLOY_SAMPLE_N or check your AWS credentials.')
#     else:
#         print(f'Running Checkov on {len(df_dep_for_checkov)} deployable templates…')

#         checkov_results = [
#             checkov_scan(row['content'])
#             for _, row in tqdm(
#                 df_dep_for_checkov.iterrows(),
#                 total=len(df_dep_for_checkov),
#                 desc='Checkov'
#             )
#         ]

#         ck_df = pd.DataFrame(checkov_results)

#         # Safety: ensure required columns always exist even if all scans errored
#         for col in ('checkov_passed', 'checkov_failed', 'checkov_fcr', 'checkov_error'):
#             if col not in ck_df.columns:
#                 ck_df[col] = None

#         df_dep_for_checkov = pd.concat(
#             [df_dep_for_checkov.reset_index(drop=True), ck_df], axis=1
#         )

#         # ── Metrics ──────────────────────────────────────────────────────────
#         valid_mask = df_dep_for_checkov['checkov_fcr'].notna()
#         n_valid    = valid_mask.sum()

#         if n_valid == 0:
#             print('⚠️  All Checkov scans returned no parseable output.')
#             print('    Checkov stdout sample:')
#             # Re-run one file in verbose mode for diagnosis
#             sample_content = df_dep_for_checkov.iloc[0]['content']
#             with tempfile.TemporaryDirectory() as t:
#                 p = pathlib.Path(t) / 'main.tf'
#                 p.write_text(sample_content)
#                 dbg = subprocess.run(
#                     ['checkov', '-f', str(p), '--framework', 'terraform', '-o', 'json'],
#                     capture_output=True, text=True, timeout=60
#                 )
#                 print('  RETURN CODE:', dbg.returncode)
#                 print('  STDOUT (first 800 chars):\n', dbg.stdout[:800])
#                 print('  STDERR (first 400 chars):\n', dbg.stderr[:400])
#         else:
#             fcr     = df_dep_for_checkov.loc[valid_mask, 'checkov_fcr'].mean()
#             avg_p   = df_dep_for_checkov.loc[valid_mask, 'checkov_passed'].mean()
#             avg_f   = df_dep_for_checkov.loc[valid_mask, 'checkov_failed'].mean()

#             print(f'\n{"─"*50}')
#             print(f'  Templates scanned          : {n_valid}')
#             print(f'  Filtered Compliance Rate   : {fcr:.1%}')
#             print(f'  (DPIaC-Eval baseline: 8.4% | target: >20%)')
#             print(f'  Avg checks passed / file   : {avg_p:.1f}')
#             print(f'  Avg checks failed / file   : {avg_f:.1f}')
#             print(f'{"─"*50}')

#         df_dep_for_checkov.to_csv(
#             DATASET_DIR / 'aws_deployable_with_checkov.csv', index=False
#         )
#         print(f'\nSaved → {DATASET_DIR / "aws_deployable_with_checkov.csv"}')


In [ ]:
# ── 12. Dataset Visualisation ─────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('IaC Benchmark Dataset – Overview', fontsize=14, fontweight='bold')

# (a) Files by repository
ax = axes[0, 0]
src = df_aws['source_slug'].str.split('/').str[-1].value_counts()
ax.barh(src.index, src.values, color='steelblue')
ax.set_title('Files per Repository (syntax-clean)')
ax.set_xlabel('Count')

# (b) Difficulty distribution
ax = axes[0, 1]
diff = df_aws['difficulty'].value_counts().sort_index()
ax.bar(diff.index.astype(str), diff.values,
       color=['#2ECC71','#3498DB','#9B59B6','#F39C12','#E74C3C'])
ax.set_title('Difficulty Distribution (DPIaC-Eval levels)')
ax.set_xlabel('Difficulty Level'); ax.set_ylabel('Count')

# (c) LOC histogram
ax = axes[1, 0]
df_aws['loc'].clip(upper=MAX_LOC).hist(bins=40, ax=ax, color='teal', edgecolor='white')
ax.set_title('Lines of Code Distribution')
ax.set_xlabel('LoC'); ax.set_ylabel('Frequency')

# (d) Cloud target pie
ax = axes[1, 1]
cloud = df_aws['primary_cloud'].value_counts()
ax.pie(cloud.values, labels=cloud.index, autopct='%1.0f%%',
       colors=['#FF9900','#4285F4','#00A1F1','#7FBA00','#EA4335'])
ax.set_title('Cloud Target Distribution')

plt.tight_layout()
plt.savefig(DATASET_DIR / 'dataset_overview.png', dpi=150)
plt.show()
print('Saved: dataset_overview.png')


In [ ]:
# !conda install -c plotly plotly-orca

In [ ]:
# ── 13. Dataset Visualisation – AWS Service Distribution & Complexity Table ───
# Saves PNGs via matplotlib.pyplot.savefig (no kaleido needed).

import hcl2, io, re, json
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — no display needed
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path
from tqdm.notebook import tqdm

DATASET_DIR.mkdir(parents=True, exist_ok=True)

# ── SERVICE_PREFIXES: covers all ~1615 resources in hashicorp/aws provider ────
SERVICE_PREFIXES = {
    # Compute
    'instance': 'EC2', 'ami': 'EC2', 'ebs': 'EC2', 'ec2': 'EC2',
    'spot': 'EC2', 'placement': 'EC2', 'key_pair': 'EC2',
    'launch': 'EC2', 'network_interface': 'EC2', 'volume': 'EC2',
    'elastic_ip': 'EC2', 'eip': 'EC2', 'fleet': 'EC2',
    # Networking / VPC
    'vpc': 'VPC', 'subnet': 'VPC', 'internet_gateway': 'VPC',
    'nat_gateway': 'VPC', 'route_table': 'VPC', 'route': 'VPC',
    'network_acl': 'VPC', 'security_group': 'VPC', 'flow_log': 'VPC',
    'vpn': 'VPC', 'transit_gateway': 'VPC', 'dx': 'Direct Connect',
    'customer_gateway': 'VPC', 'vpn_gateway': 'VPC',
    'default_vpc': 'VPC', 'default_subnet': 'VPC', 'default_route_table': 'VPC',
    'default_security_group': 'VPC', 'default_network_acl': 'VPC',
    'egress_only_internet_gateway': 'VPC', 'main_route_table_association': 'VPC',
    'network_interface_attachment': 'VPC', 'prefix_list': 'VPC',
    'managed_prefix_list': 'VPC',
    # Storage
    's3': 'S3', 'glacier': 'Glacier', 'storage_gateway': 'Storage Gateway',
    'efs': 'EFS', 'fsx': 'FSx', 'datasync': 'DataSync',
    'backup': 'Backup', 'dlm': 'DLM',
    # Database
    'rds': 'RDS', 'db': 'RDS', 'aurora': 'RDS',
    'dynamodb': 'DynamoDB', 'dax': 'DAX',
    'elasticache': 'ElastiCache', 'memorydb': 'MemoryDB',
    'redshift': 'Redshift', 'neptune': 'Neptune',
    'docdb': 'DocumentDB', 'timestream': 'Timestream',
    'keyspaces': 'Keyspaces', 'opensearch': 'OpenSearch',
    'elasticsearch': 'OpenSearch',
    # Serverless / App Integration
    'lambda': 'Lambda', 'sfn': 'Step Functions',
    'sqs': 'SQS', 'sns': 'SNS', 'eventbridge': 'EventBridge',
    'cloudwatch': 'CloudWatch', 'events': 'EventBridge',
    'scheduler': 'EventBridge', 'pipes': 'EventBridge',
    'kinesis': 'Kinesis', 'firehose': 'Kinesis Firehose',
    'msk': 'MSK', 'kafka': 'MSK',
    # API / Frontend
    'apigateway': 'API Gateway', 'api_gateway': 'API Gateway',
    'appsync': 'AppSync', 'amplify': 'Amplify',
    'cloudfront': 'CloudFront',
    # Containers
    'ecs': 'ECS', 'ecr': 'ECR', 'eks': 'EKS',
    'app_runner': 'App Runner', 'apprunner': 'App Runner', 'batch': 'Batch',
    # IAM / Security
    'iam': 'IAM', 'organizations': 'Organizations',
    'kms': 'KMS', 'secretsmanager': 'Secrets Manager',
    'acm': 'ACM', 'acmpca': 'ACM PCA',
    'cognito': 'Cognito', 'sso': 'SSO', 'ssoadmin': 'SSO',
    'identitystore': 'SSO', 'ram': 'RAM',
    # Network Security
    'waf': 'WAF', 'wafv2': 'WAF', 'shield': 'Shield',
    'guardduty': 'GuardDuty', 'inspector': 'Inspector',
    'inspector2': 'Inspector', 'macie2': 'Macie',
    'security_hub': 'Security Hub', 'securityhub': 'Security Hub',
    'detective': 'Detective', 'fms': 'FMS',
    'network_firewall': 'Network Firewall', 'networkfirewall': 'Network Firewall',
    'verified_access': 'Verified Access',
    # Load Balancing
    'lb': 'ELB', 'alb': 'ELB', 'elb': 'ELB', 'load_balancer': 'ELB',
    # DNS
    'route53': 'Route 53', 'route53resolver': 'Route 53',
    'route53recoverycontrolconfig': 'Route 53',
    'route53recoveryreadiness': 'Route 53',
    # Monitoring / Ops
    'ssm': 'SSM', 'config': 'Config', 'cloudtrail': 'CloudTrail',
    'xray': 'X-Ray', 'fis': 'FIS',
    'service_discovery': 'Cloud Map', 'servicediscovery': 'Cloud Map',
    # Automation / DevOps
    'autoscaling': 'Auto Scaling', 'autoscalingplans': 'Auto Scaling',
    'appautoscaling': 'App Auto Scaling',
    'cloudformation': 'CloudFormation',
    'codeartifact': 'CodeArtifact', 'codecommit': 'CodeCommit',
    'codebuild': 'CodeBuild', 'codedeploy': 'CodeDeploy',
    'codepipeline': 'CodePipeline', 'codestar': 'CodeStar',
    'codestarconnections': 'CodeStar', 'codestarnotifications': 'CodeStar',
    # AI / ML
    'sagemaker': 'SageMaker', 'comprehend': 'Comprehend',
    'lex': 'Lex', 'lexv2models': 'Lex',
    'rekognition': 'Rekognition', 'transcribe': 'Transcribe',
    'bedrock': 'Bedrock', 'bedrockagent': 'Bedrock',
    # Analytics
    'athena': 'Athena', 'emr': 'EMR', 'glue': 'Glue',
    'lakeformation': 'Lake Formation', 'quicksight': 'QuickSight',
    'datapipeline': 'Data Pipeline', 'cleanrooms': 'Clean Rooms',
    # Misc
    'ses': 'SES', 'sesv2': 'SES', 'pinpoint': 'Pinpoint',
    'lightsail': 'Lightsail', 'workspaces': 'WorkSpaces',
    'appconfig': 'AppConfig', 'appconfigdata': 'AppConfig',
    'servicecatalog': 'Service Catalog',
    'resourcegroups': 'Resource Groups',
    'budgets': 'Budgets', 'ce': 'Cost Explorer', 'cur': 'Cost Explorer',
    'grafana': 'Grafana', 'prometheus': 'Prometheus',
    'location': 'Location', 'opensearchserverless': 'OpenSearch',
    'verifiedpermissions': 'Cedar', 'codecatalyst': 'CodeCatalyst',
    'ivs': 'IVS', 'ivschat': 'IVS', 'chime': 'Chime',
    'dms': 'DMS', 'mgn': 'MGN',
    'mediaconvert': 'MediaConvert', 'medialive': 'MediaLive',
    'mediapackage': 'MediaPackage', 'mediatailor': 'MediaTailor',
    'iot': 'IoT', 'iotevents': 'IoT Events', 'iotanalytics': 'IoT',
    'account': 'Account', 'savingsplans': 'Savings Plans',
}

def resource_to_service(rtype: str) -> str:
    """Derive AWS service from terraform resource type using progressive prefix lookup."""
    if not rtype.startswith('aws_'):
        return 'Non-AWS'
    parts = rtype[4:].split('_')
    for length in range(min(4, len(parts)), 0, -1):
        key = '_'.join(parts[:length])
        if key in SERVICE_PREFIXES:
            return SERVICE_PREFIXES[key]
    return parts[0].upper() if parts else 'Other'


# ── 13a. Parse AST ────────────────────────────────────────────────────────────
def parse_tf_metrics(content: str) -> dict:
    metrics = {
        'resource_count': 0, 'variable_count': 0, 'output_count': 0,
        'data_source_count': 0, 'module_count': 0, 'resource_types': [],
    }
    def _unwrap(v):
        return v[0] if isinstance(v, list) and len(v) == 1 else v

    try:
        ast = hcl2.load(io.StringIO(content))
        for block in ast.get('resource', []):
            for rtype, instances in block.items():
                metrics['resource_count'] += len(instances) if isinstance(instances, list) else 1
                metrics['resource_types'].append(rtype)
        for block in ast.get('variable', []):
            metrics['variable_count'] += len(block)
        for block in ast.get('output', []):
            metrics['output_count'] += len(block)
        for block in ast.get('data', []):
            metrics['data_source_count'] += len(block)
        for block in ast.get('module', []):
            metrics['module_count'] += len(block)
    except Exception:
        metrics['resource_count']    = len(re.findall(r'^resource\s+"[^"]*"', content, re.MULTILINE))
        metrics['variable_count']    = len(re.findall(r'^variable\s+"[^"]*"', content, re.MULTILINE))
        metrics['output_count']      = len(re.findall(r'^output\s+"[^"]*"',   content, re.MULTILINE))
        metrics['data_source_count'] = len(re.findall(r'^data\s+"[^"]*"',     content, re.MULTILINE))
        metrics['module_count']      = len(re.findall(r'^module\s+"[^"]*"',   content, re.MULTILINE))
        metrics['resource_types']    = re.findall(r'^resource\s+"([^"]*)"',   content, re.MULTILINE)
    return metrics

print('Parsing HCL2 AST…')
metrics_list = [parse_tf_metrics(c) for c in tqdm(df_aws['content'], desc='AST parse')]
df_metrics   = pd.DataFrame(metrics_list)
df_viz       = pd.concat([df_aws.reset_index(drop=True), df_metrics], axis=1)

# ── 13b. Service counts ───────────────────────────────────────────────────────
all_rtypes     = [rt for sub in df_viz['resource_types'] for rt in sub]
service_series = pd.Series([resource_to_service(rt) for rt in all_rtypes])
service_counts = service_series.value_counts()

# Unrecognised check
unrecognised = [rt for rt in all_rtypes if resource_to_service(rt) not in SERVICE_PREFIXES.values()
                and not rt.startswith('aws_')]
print(f'Unrecognised (non-aws_ prefix): {len(unrecognised)}')

TOP_N        = 20
top_services = service_counts.head(TOP_N)
other_count  = service_counts.iloc[TOP_N:].sum()
if other_count > 0:
    top_services = pd.concat([top_services, pd.Series({'Other': other_count})])

# ── 13c. Complexity table ─────────────────────────────────────────────────────
DIFFICULTY_LABELS = {1:'L1 (<50 LoC)', 2:'L2 (50–100)', 3:'L3 (100–150)',
                     4:'L4 (150–200)', 5:'L5 (>200 LoC)'}
df_viz['diff_label'] = df_viz['difficulty'].map(DIFFICULTY_LABELS)

complexity_table = (
    df_viz.groupby('diff_label')
    .agg(
        files         =('dest_file',        'count'),
        mean_loc      =('loc',               'mean'),
        median_loc    =('loc',               'median'),
        mean_tokens   =('tokens',            'mean'),
        mean_resources=('resource_count',    'mean'),
        mean_variables=('variable_count',    'mean'),
        mean_outputs  =('output_count',      'mean'),
        mean_data_src =('data_source_count', 'mean'),
        mean_modules  =('module_count',      'mean'),
    )
    .round(1)
    .reindex([DIFFICULTY_LABELS[i] for i in range(1,6)])
    .reset_index()
)
complexity_table.to_csv(DATASET_DIR / 'dataset_complexity.csv', index=False)
print(f'Saved: dataset_complexity.csv')

# ── helper: save fig as png ───────────────────────────────────────────────────
def savefig(path: str, dpi: int = 150):
    plt.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'Saved: {path}')

# ── Figure 1: AWS Service Distribution ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 10))
colors  = plt.cm.tab20.colors
bars    = ax.barh(top_services.index[::-1], top_services.values[::-1],
                  color=colors[:len(top_services)])
ax.bar_label(bars, padding=4, fontsize=9)
ax.set_xlabel('Resource Count', fontsize=11)
ax.set_title(f'AWS Service Distribution\n'
             f'{len(df_viz):,} templates · {len(all_rtypes):,} total resources',
             fontsize=13, fontweight='bold')
ax.set_xlim(0, top_services.max() * 1.18)
ax.grid(axis='x', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
savefig(str(DATASET_DIR / 'aws_service_distribution.png'))

# ── Figure 2: Complexity table as matplotlib table ───────────────────────────
col_labels = ['Difficulty', '#Files', 'Mean\nLoC', 'Median\nLoC',
              'Mean\nTokens', '#Resources', '#Variables',
              '#Outputs', '#DataSrc', '#Modules']
cell_data = []
for _, r in complexity_table.iterrows():
    cell_data.append([
        r['diff_label'], int(r['files']),
        f"{r['mean_loc']:.0f}", f"{r['median_loc']:.0f}",
        f"{r['mean_tokens']:.0f}",
        f"{r['mean_resources']:.1f}", f"{r['mean_variables']:.1f}",
        f"{r['mean_outputs']:.1f}", f"{r['mean_data_src']:.1f}",
        f"{r['mean_modules']:.1f}",
    ])

fig, ax = plt.subplots(figsize=(14, 3))
ax.axis('off')
tbl = ax.table(
    cellText   = cell_data,
    colLabels  = col_labels,
    loc        = 'center',
    cellLoc    = 'center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1, 1.8)

# Header styling
for j in range(len(col_labels)):
    tbl[0, j].set_facecolor('#2c3e50')
    tbl[0, j].set_text_props(color='white', fontweight='bold')

# Alternating row shading
for i in range(1, len(cell_data) + 1):
    for j in range(len(col_labels)):
        tbl[i, j].set_facecolor('#f0f4f8' if i % 2 == 0 else 'white')

ax.set_title('Template Complexity by Difficulty Level',
             fontsize=12, fontweight='bold', pad=12)
savefig(str(DATASET_DIR / 'complexity_table.png'))

# ── Figure 3: Resource count distribution by difficulty (box plot) ────────────
ordered_labels = [DIFFICULTY_LABELS[i] for i in range(1, 6)
                  if DIFFICULTY_LABELS[i] in df_viz['diff_label'].values]
data_by_diff   = [df_viz.loc[df_viz['diff_label'] == l, 'resource_count'].values
                  for l in ordered_labels]

fig, ax = plt.subplots(figsize=(10, 5))
bp = ax.boxplot(data_by_diff, labels=ordered_labels, patch_artist=True,
                medianprops=dict(color='#e74c3c', linewidth=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.4))
colors_box = plt.cm.Blues(np.linspace(0.35, 0.75, len(ordered_labels)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)
ax.set_xlabel('Difficulty Level', fontsize=11)
ax.set_ylabel('# Resources', fontsize=11)
ax.set_title('Resource Count Distribution by Difficulty\n(box=IQR, whiskers=1.5×IQR)',
             fontsize=12, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.spines[['top','right']].set_visible(False)
savefig(str(DATASET_DIR / 'resource_distribution.png'))

print('\nAll figures saved to:', DATASET_DIR)
